<div align="center">

  # <span style="color:orange;">*NEO DATAFEST 2026*</span>
  #### <span style="color:Blue;">Prediksi Risiko Keterlambatan Pengiriman Menggunakan Graph Convolutional Network dengan Optimasi Parameter</span>

---

</div>

*Catatan analisis yang dibuat dengan penuh dedikasi oleh:*

1. Himam Bashiran
2. Afan Ramadhan
3. Muhammad Naufal

# **1. Setup**

Bagian ini menyiapkan lingkungan komputasi beserta seluruh konfigurasi awal sebelum pemrosesan data dan pelatihan model dijalankan. Penyiapan tersebut menjamin bahwa eksperimen prediksi penyebaran kebakaran hutan berlangsung secara konsisten, terstruktur, dan dapat diulang pada perangkat keras yang tersedia.

* Memasang serta mengimpor pustaka komputasi numerik, analisis geospasial, visualisasi ilmiah, dan kerangka kerja *deep learning* maupun *gradient boosting*.
* Mengunci *random seed* global agar pembagian data, pengacakan sampel, dan inisialisasi bobot bersifat deterministik.
* Mendeteksi ketersediaan akselerator GPU dan mengatur konfigurasi pelatihan multi GPU secara otomatis.
* Menetapkan parameter dasar eksperimen, direktori penyimpanan keluaran, serta standar visualisasi grafik publikasi.

### **Import Library**

Subbagian ini memasang dan mengimpor seluruh pustaka Python yang diperlukan sepanjang alur eksperimen, meliputi PyTorch dan Segmentation Models PyTorch untuk pembangunan model spasial, XGBoost untuk model pembanding berbasis pohon keputusan, serta GeoPandas, Contextily, Shapely, Matplotlib, dan Seaborn untuk pemetaan geospasial maupun penyusunan grafik ilmiah.

In [ ]:
!pip install -q segmentation_models_pytorch pytorch-optimizer xgboost geopandas contextily shapely adjustText

In [ ]:
import os, gc
import ctypes
import numpy as np
import pandas as pd
import seaborn as sns
import pickle, shutil
import contextily as cx
import geopandas as gpd
from pathlib import Path
from functools import partial
from google.colab import drive
import matplotlib.pyplot as plt
import json, time, math, random
from scipy import ndimage as ndi
from shapely.prepared import prep
from adjustText import adjust_text
from pytorch_optimizer import SOAP
from matplotlib.colors import LogNorm
from matplotlib.collections import PatchCollection
from matplotlib.patches import Patch, Polygon as MplPolygon
from matplotlib.ticker import FuncFormatter, MaxNLocator
from shapely.geometry import Polygon as ShapelyPolygon
from scipy.stats import friedmanchisquare, wilcoxon
import torch
import torch.nn as nn
import xgboost as xgb
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import average_precision_score, precision_recall_curve, brier_score_loss
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

### **Global Configuration**

Subbagian ini mendefinisikan fungsi penguncian *random seed* demi keterulangan hasil, mendeteksi jumlah perangkat CUDA untuk dukungan komputasi paralel, menetapkan ukuran *patch* sebesar 64 kali 64 piksel pada resolusi 375 meter untuk tiga horizon prediksi 72 jam, serta membentuk direktori penyimpanan hasil maupun *checkpoint* beserta standar tampilan grafik.

In [ ]:
# Menetapkan nilai random seed global agar seluruh eksperimen dapat direproduksi secara konsisten.
SEED = 42
def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
seed_all(SEED)

In [ ]:
# Mendeteksi ketersediaan akselerator GPU dan mengaktifkan mode paralel apabila terpasang lebih dari satu perangkat.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_MULTI_GPU = N_GPUS > 1
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
print(f'Device: {DEVICE} | GPUs: {N_GPUS} | Multi-GPU (DataParallel): {USE_MULTI_GPU}')
# Mengatur konfigurasi tambahan.
SMOKE = False
RETRAIN = True
H = 3
PATCH_SIZE = 64
RESOLUTION_M = 375.0
BUFFER_RADII = (4, 6, 8)
EPOCHS = 10 if SMOKE else 100
BATCH_SIZE = 32 * max(1, N_GPUS)
LR = 1e-4
POS_W_MAX = 5.0

In [ ]:
# Menyiapkan direktori work untuk menyimpan tabel hasil evaluasi dan berkas bobot model.
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
RESULT_DIR = ROOT / 'results'
CKPT_DIR = ROOT / 'checkpoints'
for d in [RESULT_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
test_loaded = False

In [ ]:
sns.set_theme(
    style="ticks",
    context="paper",
     font="DejaVu Sans",
    font_scale=1.1,
    rc={
        "figure.dpi": 150,
        "savefig.dpi": 600,
        "savefig.bbox": "tight",
        "axes.linewidth": 0.8,
        "axes.labelsize": 10,
        "axes.titlesize": 11,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "lines.linewidth": 1.8,
        "lines.markersize": 5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)
SCIENTIFIC_PALETTE = [
    "#0072B2",  # Blue
    "#E69F00",  # Orange
    "#009E73",  # Green
    "#D55E00",  # Red
    "#CC79A7",  # Purple
    "#8C564B",  # Brown
]
SPLIT_PALETTE = {'Data Latih': SCIENTIFIC_PALETTE[0], 'Data Validasi': SCIENTIFIC_PALETTE[1]}
IGBP_LABELS_ID = {
    2: 'Hutan Berdaun Lebar Hijau', 4: 'Hutan Berdaun Lebar Gugur', 5: 'Hutan Campuran',
    6: 'Semak Belukar Tertutup', 7: 'Semak Belukar Terbuka', 8: 'Sabana Berkayu',
    9: 'Sabana', 10: 'Padang Rumput', 11: 'Lahan Basah / Rawa Gambut',
    12: 'Lahan Pertanian', 13: 'Permukiman & Terbangun', 14: 'Mosaik Pertanian–Vegetasi',
    16: 'Lahan Terbuka', 17: 'Badan Air',
}
sns.set_palette(SCIENTIFIC_PALETTE)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

# **2. Data Preparation**

Bagian ini menyiapkan himpunan data penyebaran kebakaran hutan di Pulau Kalimantan melalui proses pengunduhan dan pemuatan berkas tensor ke dalam memori. Pemeriksaan integritas struktur data langsung dijalankan pada tahap ini guna memastikan kebenaran dimensi, rentang koordinat, dan fitur fisika sebelum analisis lebih lanjut.

* Mengunduh himpunan data dari Kaggle secara otomatis atau beralih ke direktori data lokal apabila akses daring tidak tersedia.
* Memuat arsip terkompresi data latih dan data validasi sambil mengonversi tensor `float16` menjadi `float32` secara efisien di memori.
* Memverifikasi dimensi sampel, proporsi piksel target api baru, keabsahan kotak pembatas geografis, serta rentang nilai sudut fisika.

#### Download Dataset from Kaggle

Subsubbagian ini mengunduh himpunan data kebakaran hutan Kalimantan melalui pustaka `kagglehub` dan menetapkan jalur direktori lokal sebagai cadangan apabila pengunduhan gagal.

In [ ]:
# Mengunduh dataset kebakaran hutan Kalimantan dari Kaggle atau memakai direktori lokal sebagai cadangan.
try:
    import kagglehub
    kaggle_data_path = kagglehub.dataset_download('sanantaaulgard/borneo-island-wildfire-spread-dataset')
    DATA_DIR = Path(kaggle_data_path)
except Exception:
    DATA_DIR = ROOT / 'data'

In [ ]:
# Memuat berkas data NPZ per partisi dan mengonversi larik setengah presisi ke float32 untuk stabilitas numerik.
def load_npz(split, max_n=None):
    path = DATA_DIR / f'{split}.npz'
    keys = ['x', 'y', 'y_new', 'valid', 'state_t', 'ca_phys', 'forcing_g', 'scalars', 'bbox', 'center_coord', 'date', 'timestamps', 'event_id', 'province']
    d = {}
    with np.load(path) as raw:
        for k in keys:
            if k in raw:
                arr = raw[k] if max_n is None else raw[k][:max_n]
                d[k] = arr.astype(np.float32, copy=False) if arr.dtype == np.float16 else arr
    # Menurunkan masker area terbakar awal dengan menyisihkan piksel terbakar baru dari target kumulatif pertama.
    if 'burned_t' not in d and 'y' in d and 'y_new' in d:
        d['burned_t'] = d['y'][:, 0] & ~d['y_new'][:, 0]
    return d

### **Load Train & Val Dataset**

Subbagian ini memuat data latih sebanyak 3.699 sampel dan data validasi sebanyak 790 sampel ke dalam memori menggunakan fungsi `load_npz`, lalu memverifikasi bahwa proporsi piksel terbakar baru pada data latih tercatat sebesar 0,3131 persen sekaligus memastikan validitas konsistensi masker awal, kotak pembatas spasial, dan rentang sudut angin maupun lereng dalam satuan radian.

In [ ]:
# Memuat himpunan data latih serta validasi sesuai batas jumlah sampel mode eksperimen.
max_tr = 100 if SMOKE else None
max_va = 32 if SMOKE else None
train_d = load_npz('train', max_n=max_tr)
val_d = load_npz('val', max_n=max_va)

In [ ]:
dim_summary = pd.DataFrame([
    {'Split': 'Train', 'Samples': len(train_d['x']), 'x_shape': str(train_d['x'].shape), 'y_new_shape': str(train_d['y_new'].shape)},
    {'Split': 'Val', 'Samples': len(val_d['x']), 'x_shape': str(val_d['x'].shape), 'y_new_shape': str(val_d['y_new'].shape)}
])
dim_summary

In [ ]:
# Memverifikasi konsistensi masker area terbakar awal terhadap target perluasan api dan menghitung prevalensi kelas positif.
b0 = train_d['y'][:, 0] & ~train_d['y_new'][:, 0]
assert (b0 == train_d['burned_t']).all()
assert not (train_d['y_new'] & train_d['burned_t'][:, None]).any()
y_new_prev = train_d['y_new'].sum() / (train_d['y_new'].size) * 100

print(f'proporsi y_new: {y_new_prev:.4f}%')

In [ ]:
lon_span = train_d['bbox'][:, 2] - train_d['bbox'][:, 0]
lat_span = train_d['bbox'][:, 3] - train_d['bbox'][:, 1]
assert (lon_span > 0).all() and (lat_span > 0).all()
assert len(train_d['date']) == len(train_d['x'])

In [ ]:
th_w = train_d['ca_phys'][:, 1]
th_s = train_d['ca_phys'][:, 3]
assert (th_w >= 0).all() and (th_w <= 2 * math.pi + 1e-4).all()
assert (th_s >= 0).all() and (th_s <= 2 * math.pi + 1e-4).all()

# **3. Exploratory Data Analysis (EDA)**

Bagian ini mengeksplorasi pola spasial, dinamika musiman, ketidakseimbangan kelas, dan faktor pendorong lingkungan maupun antropogenik pada 4.489 *patch* kebakaran di Pulau Kalimantan. Temuan empiris pada bagian ini menjadi landasan dalam merancang domain evaluasi, strategi pembobotan sampel, serta desain arsitektur berbasis fisika.

* Menyiapkan utilitas denormalisasi fitur serta metadata gabungan dari himpunan data latih dan data validasi beserta peta batas administratif provinsi.
* Memetakan sebaran spasial intensitas kebakaran di seluruh Kalimantan, tren musiman bulanan, serta ketimpangan ekstrem antara kelas terbakar dan tidak terbakar.
* Memvisualisasikan contoh penjalaran api selama 72 jam bersama area yang terhalang awan atau badan air.
* Mengkaji keterkaitan laju penjalaran api dengan tutupan lahan dominan, jarak jalan dan deforestasi, arah adveksi angin, fase iklim ONI maupun DMI, serta dominasi lahan gambut di setiap provinsi.

### **Plot Configuration**

Subbagian ini menyusun fungsi pengembalian skala nilai fitur ke satuan fisik asli, memuat geometri batas provinsi Kalimantan, membentuk tabel ringkasan statistik `eda_df` dari seluruh 4.489 sampel latih dan validasi, serta menyediakan fungsi pembantu peta geografis lengkap dengan *basemap*, skala jarak, dan penunjuk arah utara.

In [ ]:
# Mengembalikan nilai fitur kontinu ke skala fisik aslinya berdasarkan statistik normalisasi dataset.
def unnormalize_channel(arr, ch_name, ch_stats):
    st = ch_stats[ch_name]
    if st.get('is_categorical', False) or st.get('is_binary_or_trig', False):
        return arr.astype(np.float32)
    raw = arr.astype(np.float32) * float(st['std']) + float(st['mean'])
    return np.expm1(np.clip(raw, -20.0, 20.0)) if st.get('apply_log1p', False) else raw

def add_lonlat(ax):
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{abs(x):.2f}°{'BT' if x >= 0 else 'BB'}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{abs(y):.2f}°{'LU' if y >= 0 else 'LS'}"))
    ax.grid(color='grey', linewidth=0.4, alpha=0.5, linestyle='--')

def add_north_arrow(ax, x=0.94, y=0.92):
    ax.annotate('U', xy=(x, y), xytext=(x, y - 0.11), xycoords='axes fraction',
                ha='center', va='center', fontsize=9, fontweight='bold',
                arrowprops=dict(facecolor='black', edgecolor='white', width=3, headwidth=9))

def add_scalebar(ax, x=0.04, y=0.05):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    km_per_deg = 111.32 * np.cos(np.radians((ymin + ymax) / 2.0))
    width_km = max((xmax - xmin) * km_per_deg, 1e-3)
    length_km = next((n for n in (500, 200, 100, 50, 20, 10, 5, 2, 1, 0.5) if n <= width_km / 4.0), 0.5)
    length_deg = length_km / km_per_deg
    x0, y0, height = xmin + x * (xmax - xmin), ymin + y * (ymax - ymin), (ymax - ymin) * 0.014
    for i in range(2):
        ax.add_patch(plt.Rectangle((x0 + i * length_deg / 2.0, y0), length_deg / 2.0, height,
                                   facecolor='black' if i == 0 else 'white', edgecolor='black', linewidth=0.7, zorder=10))
    for frac in (0.0, 0.5, 1.0):
        ax.text(x0 + frac * length_deg, y0 + height * 1.6, f'{length_km * frac:g}', ha='center', va='bottom',
                fontsize=7, bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=0.8), zorder=10)
    ax.text(x0 + length_deg + (xmax - xmin) * 0.01, y0, 'km', va='bottom', fontsize=7, zorder=10)

def add_esri_basemap(ax, extent, alpha=1.0, attribution_size=6):
    if cx is None:
        return
    try:
        zoom = cx.tile._calculate_zoom(extent[0], extent[2], extent[1], extent[3]) + 1
        cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.Esri.WorldShadedRelief,
                       zoom=zoom, alpha=alpha, attribution_size=attribution_size)
    except Exception:
        pass

# Menyiapkan kanvas basemap Pulau Kalimantan lengkap dengan batas provinsi, koordinat, arah utara, dan batang skala.
def setup_kalimantan_map(ax, prov_gdf, lw=0.9, attribution_size=6, north_xy=(0.94, 0.92), scale_xy=(0.04, 0.05)):
    xmin, ymin, xmax, ymax = prov_gdf.total_bounds
    pad = 0.45
    extent = [xmin - pad, xmax + pad, ymin - pad, ymax + pad]
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_aspect('equal')
    add_esri_basemap(ax, extent, alpha=0.65, attribution_size=attribution_size)
    prov_gdf.plot(ax=ax, facecolor='none', edgecolor='#1a202c', linewidth=lw, zorder=4)
    add_lonlat(ax)
    add_north_arrow(ax, x=north_xy[0], y=north_xy[1])
    add_scalebar(ax, x=scale_xy[0], y=scale_xy[1])
    ax.set_xlabel('Bujur')
    ax.set_ylabel('Lintang')

def save_and_show(fig, filename):
    plt.show()
    plt.close(fig)

# Menghaluskan batas masker biner menggunakan interpolasi bilinear dan Gaussian agar visualisasi peta lebih rapi.
def smooth_mask(mask_2d, scale=4, sigma=1.3):
    return ndi.gaussian_filter(ndi.zoom(mask_2d.astype(np.float32), scale, order=1), sigma=sigma) >= 0.45

In [ ]:
# Memuat geometri batas administratif provinsi Kalimantan dari berkas lokal, repositori GADM, atau aproksimasi titik data.
def load_kalimantan_provinces(df_fallback):
    candidates = [
        ROOT / 'output/gadm/kalimantan_provinces.parquet',
        DATA_DIR / 'kalimantan_provinces.parquet',
        RESULT_DIR / 'kalimantan_provinces.parquet',
    ]
    for p in candidates:
        if p.exists():
            return gpd.read_parquet(p).to_crs('EPSG:4326')
    try:
        gdf = gpd.read_file('https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_IDN_1.json.zip')
        kal = gdf[gdf['NAME_1'].str.contains('Kalimantan', na=False)].copy()
        kal['province_name'] = kal['NAME_1'].str.replace('Kalimantan', 'Kalimantan ', regex=False).str.strip()
        kal = kal[['province_name', 'geometry']].to_crs('EPSG:4326')
        kal.to_parquet(RESULT_DIR / 'kalimantan_provinces.parquet')
        return kal
    except Exception:
        pts = gpd.GeoDataFrame(
            {'province_name': df_fallback['province']},
            geometry=gpd.points_from_xy(df_fallback['lon'], df_fallback['lat']),
            crs='EPSG:4326',
        )
        return pts.dissolve(by='province_name').convex_hull.reset_index(name='geometry')

def _masked_mean(arr, mask):
    cnt = mask.sum(axis=(1, 2))
    return np.where(cnt > 0, (arr * mask).sum(axis=(1, 2)) / np.maximum(cnt, 1), arr.mean(axis=(1, 2)))

# Menambah atribut spasial, temporal, iklim global, dan tutupan lahan dari data latih serta validasi ke dalam tabel ringkasan EDA.
def build_eda_data(train_d, val_d):
    meta = json.loads((DATA_DIR / 'metadata.json').read_text(encoding='utf-8'))
    ch_idx = {name: i for i, name in enumerate(meta['core_32_channels'])}
    ch_stats = meta['normalization_stats']['channels']
    raw_arrays = {'Data Latih': train_d, 'Data Validasi': val_d}
    frames = []
    for sp, d in raw_arrays.items():
        x, y, yn = d['x'], d['y'], d['y_new']
        v = d['valid'] > 0
        v_h1, v_h2, v_h3 = v[:, 0], v[:, 1], v[:, 2]
        dates = pd.to_datetime(d['date'].astype(str))
        coords = d['center_coord'] if 'center_coord' in d else np.stack([(d['bbox'][:, 0] + d['bbox'][:, 2]) / 2, (d['bbox'][:, 1] + d['bbox'][:, 3]) / 2], axis=1)
        peat = unnormalize_channel(x[:, ch_idx['peat']], 'peat', ch_stats)
        dist_rd = unnormalize_channel(x[:, ch_idx['dist_road']], 'dist_road', ch_stats)
        floss = unnormalize_channel(x[:, ch_idx['forest_loss_cum']], 'forest_loss_cum', ch_stats)
        lc_arr = np.rint(x[:, ch_idx['landcover_idx']]).astype(np.int32)
        dom_lc = np.array([
            int(np.bincount((lc_arr[i][v_h3[i]] if v_h3[i].any() else lc_arr[i].ravel()).clip(min=0)).argmax())
            for i in range(len(y))
        ])
        yn_px = [(yn[:, h] & v[:, h]).sum(axis=(1, 2)) for h in range(H)]
        peat_frac = np.where(v_h3.any(axis=(1, 2)), _masked_mean(peat > 0.5, v_h3), 0.0)
        floss_frac = np.where(v_h3.any(axis=(1, 2)), _masked_mean(floss > 0.5, v_h3), 0.0)
        frames.append(pd.DataFrame({
            'split': sp,
            'idx': np.arange(len(y)),
            'event_id': d['event_id'].astype(np.int32),
            'date': dates,
            'year': dates.year,
            'month': dates.month,
            'province': d['province'].astype(str),
            'lon': coords[:, 0].astype(np.float32),
            'lat': coords[:, 1].astype(np.float32),
            'oni': d['scalars'][:, 0].astype(np.float32),
            'dmi': d['scalars'][:, 1].astype(np.float32),
            'init_fire_px': ((d['state_t'][:, 1] > 0.5) & v_h1).sum(axis=(1, 2)),
            'y_h1_px': (y[:, 0] & v_h1).sum(axis=(1, 2)),
            'y_h2_px': (y[:, 1] & v_h2).sum(axis=(1, 2)),
            'y_h3_px': (y[:, 2] & v_h3).sum(axis=(1, 2)),
            'yn_h1_px': yn_px[0],
            'yn_h2_px': yn_px[1],
            'yn_h3_px': yn_px[2],
            'yn_total_px': yn_px[0] + yn_px[1] + yn_px[2],
            'valid_h3_frac': v_h3.mean(axis=(1, 2)),
            'peat_frac': peat_frac,
            'is_peat_dom': np.where(peat_frac >= 0.5, 'Lahan Gambut', 'Lahan Non-Gambut'),
            'dist_road_mean': _masked_mean(dist_rd, v_h3),
            'floss_frac': floss_frac,
            'dom_lc': dom_lc,
        }))
    return pd.concat(frames, ignore_index=True), raw_arrays

eda_df, eda_arrays = build_eda_data(train_d, val_d)
prov_gdf = load_kalimantan_provinces(eda_df)
print(f'Total patch EDA termuat: {len(eda_df)}')

#### Spatial Distribution of Wildfire Spread Intensity Across Kalimantan

In [ ]:
# Memetakan akumulasi intensitas piksel terbakar baru pada grid heksagonal di seluruh Pulau Kalimantan.
def plot_spatial_hexbin(df, prov_gdf):
    fig, ax = plt.subplots(figsize=(9.5, 8.2))
    setup_kalimantan_map(ax, prov_gdf, lw=1.15, attribution_size=7, north_xy=(0.94, 0.93), scale_xy=(0.04, 0.05))
    xmin, ymin, xmax, ymax = prov_gdf.total_bounds
    kal_union = prov_gdf.geometry.union_all()
    kal_prep = prep(kal_union)
    r, angles = 0.08, np.linspace(0, 2 * np.pi, 7)[:-1]
    dx, dy = 1.5 * r, np.sqrt(3.0) * r
    lons, lats, weights = df['lon'].to_numpy(), df['lat'].to_numpy(), df['yn_total_px'].to_numpy() + 1.0
    hex_polys, hex_vals = [], []
    for ci, cx_pt in enumerate(np.arange(xmin - r, xmax + r, dx)):
        y_off = (dy / 2.0) if (ci % 2 == 1) else 0.0
        for cy_pt in np.arange(ymin - r + y_off, ymax + r, dy):
            poly = ShapelyPolygon([(cx_pt + r * np.cos(a), cy_pt + r * np.sin(a)) for a in angles])
            if not kal_prep.intersects(poly):
                continue
            clipped = poly if kal_prep.contains(poly) else poly.intersection(kal_union)
            if clipped.is_empty:
                continue
            in_hex = ((lons - cx_pt) ** 2 + (lats - cy_pt) ** 2) <= r ** 2
            val = 1.0 + (float(weights[in_hex].sum()) if np.any(in_hex) else 0.0)
            geoms = [clipped] if clipped.geom_type == 'Polygon' else (list(clipped.geoms) if clipped.geom_type == 'MultiPolygon' else [])
            for g in geoms:
                hex_polys.append(MplPolygon(list(g.exterior.coords), closed=True))
                hex_vals.append(val)
    hex_vals_arr = np.array(hex_vals, dtype=np.float64)
    pc = PatchCollection(hex_polys, cmap='magma_r', norm=LogNorm(vmin=1.0, vmax=max(float(hex_vals_arr.max()), 10.0)),
                         edgecolor='none', linewidths=0, alpha=0.88, zorder=5)
    pc.set_array(hex_vals_arr)
    ax.add_collection(pc)
    fig.colorbar(pc, ax=ax, fraction=0.036, pad=0.02).set_label('Intensitas Kumulatif Area Terbakar Baru (log10)', fontsize=9)
    for _, row in prov_gdf.iterrows():
        pt = row.geometry.representative_point()
        ax.text(pt.x, pt.y, str(row['province_name']).replace('Kalimantan ', 'Kal. '),
                fontsize=8, fontweight='bold', ha='center', va='center', color='#111111',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='#cbd5e0', alpha=0.85), zorder=7)
    ax.set_title('Distribusi Spasial Intensitas Penjalaran Kebakaran di Kalimantan', pad=10, fontweight='bold')

In [ ]:
plot_spatial_hexbin(eda_df, prov_gdf)

Grafik heksagonal di atas memetakan akumulasi intensitas piksel terbakar baru pada skala logaritmik di seluruh daratan Pulau Kalimantan. Persebaran kebakaran dengan intensitas tertinggi terkonsentrasi kuat di bagian selatan yaitu Kalimantan Selatan dan selatan Kalimantan Tengah, serta memanjang di pesisir barat dan selatan Kalimantan Barat, sedangkan beberapa titik berintensitas tinggi di Kalimantan Timur dan Kalimantan Utara tampak lebih terpisah secara spasial.

#### Monthly Seasonality of Wildfire Occurrence and Spread Severity

In [ ]:
# Membandingkan proporsi bulanan kejadian kebakaran per event terhadap tren rata-rata luas area terbakar baru.
def plot_monthly_seasonality(df):
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des']
    rows = [
        {'Bulan': month_names[m - 1], 'Partisi': sp, 'Proporsi (%)': (sub['month'] == m).sum() / max(len(sub), 1) * 100.0}
        for sp in ['Data Latih', 'Data Validasi']
        for sub in [df[df['split'] == sp]]
        for m in range(1, 13)
    ]
    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    sns.barplot(data=pd.DataFrame(rows), x='Bulan', y='Proporsi (%)', hue='Partisi', palette=SPLIT_PALETTE, ax=ax, edgecolor='white', linewidth=0.5)
    ax.axvspan(6.5, 9.5, color=SCIENTIFIC_PALETTE[2], alpha=0.15, zorder=0, label='Puncak Musim Kemarau (Agu–Okt)')
    ax2 = ax.twinx()
    ax2.spines['top'].set_visible(False)
    overall_m = df.groupby('month')['yn_total_px'].mean().reindex(range(1, 13), fill_value=0.0)
    ax2.plot(np.arange(12), overall_m.to_numpy(), color=SCIENTIFIC_PALETTE[3], marker='o', linewidth=2.0, label='Area Terbakar Baru')
    ax2.set_ylabel('Rata-rata Area Terbakar Baru', color=SCIENTIFIC_PALETTE[3])
    ax2.tick_params(axis='y', colors=SCIENTIFIC_PALETTE[3])
    ax2.set_ylim(0, max(overall_m.max() * 1.25, 1.0) * 1.05)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper left', frameon=True, facecolor='white', edgecolor='#e2e8f0')
    ax.set_title('Tren Musiman Kejadian dan Tingkat Keparahan Kebakaran Bulanan', fontweight='bold')
    ax.set_xlabel('Bulan Kalender')
    ax.set_ylabel('Proporsi (%)')
    ax.set_ylim(0, ax.get_ylim()[1] * 1.05)
    ax.grid(axis='y', linestyle='--', alpha=0.35)

In [ ]:
plot_monthly_seasonality(eda_df)

Grafik gabungan di atas membandingkan proporsi bulanan kejadian kebakaran pada data latih dan data validasi melalui diagram batang terhadap rata-rata luas area terbakar baru pada garis berwarna oranye. Frekuensi kejadian kebakaran meningkat tajam pada puncak musim kemarau dari Agustus hingga Oktober dengan proporsi tertinggi di bulan September mencapai sekitar 57 persen pada data latih dan 59 persen pada data validasi, sedangkan tingginya nilai rata-rata luas area terbakar pada bulan Februari dan November mengindikasikan adanya sejumlah kejadian ekstrem berskala besar di luar musim puncak kemarau.

#### Extreme Class Imbalance Analysis Across Multi-Horizon Fire Targets

In [ ]:
# Memvisualisasikan rasio class imbalance ekstrem pada target piksel terbakar baru dan target kumulatif antarhorizon.
def plot_target_imbalance_2x1(raw_arrays):
    targets = [('y_new', 'Target Piksel Terbakar Baru (y_new)'), ('y', 'Target Area Terbakar Kumulatif (y)')]
    horizons = [('t+1 (24 jam)', 0), ('t+2 (48 jam)', 1), ('t+3 (72 jam)', 2)]
    records = []
    for t_key, t_label in targets:
        tot_pos, tot_neg = 0, 0
        for h_name, h_i in horizons:
            pos = sum(int(((d[t_key][:, h_i] > 0) & (d['valid'][:, h_i] > 0)).sum()) for d in raw_arrays.values())
            neg = sum(int((~(d[t_key][:, h_i] > 0) & (d['valid'][:, h_i] > 0)).sum()) for d in raw_arrays.values())
            tot = max(pos + neg, 1)
            tot_pos, tot_neg = tot_pos + pos, tot_neg + neg
            records.extend([
                {'Target': t_label, 'Horizon': h_name, 'Kelas': 'Tidak Terbakar (0)', 'Jumlah': neg, 'Persen': neg / tot * 100.0},
                {'Target': t_label, 'Horizon': h_name, 'Kelas': 'Terbakar (1)', 'Jumlah': pos, 'Persen': pos / tot * 100.0},
            ])
        gtot = max(tot_pos + tot_neg, 1)
        records.extend([
            {'Target': t_label, 'Horizon': 'Keseluruhan', 'Kelas': 'Tidak Terbakar (0)', 'Jumlah': tot_neg, 'Persen': tot_neg / gtot * 100.0},
            {'Target': t_label, 'Horizon': 'Keseluruhan', 'Kelas': 'Terbakar (1)', 'Jumlah': tot_pos, 'Persen': tot_pos / gtot * 100.0},
        ])
    imb_df = pd.DataFrame(records)
    fig, axes = plt.subplots(2, 1, figsize=(8.8, 7.6), sharex=True)
    cls_palette = {'Tidak Terbakar (0)': SCIENTIFIC_PALETTE[1], 'Terbakar (1)': SCIENTIFIC_PALETTE[3]}
    h_order = ['t+1 (24 jam)', 't+2 (48 jam)', 't+3 (72 jam)', 'Keseluruhan']
    c_order = ['Tidak Terbakar (0)', 'Terbakar (1)']
    for ax, (_, t_label) in zip(axes, targets):
        sub = imb_df[imb_df['Target'] == t_label]
        sns.barplot(data=sub, x='Horizon', y='Jumlah', hue='Kelas', palette=cls_palette, ax=ax, edgecolor='white', linewidth=0.6)
        ax.set_yscale('log')
        ax.set_ylim(1e3, sub['Jumlah'].max() * 14.0)
        ax.set_ylabel('Jumlah Piksel Valid (skala log)')
        ax.grid(axis='y', linestyle='--', alpha=0.35)
        for c_i, cls_name in enumerate(c_order):
            c_sub = sub[sub['Kelas'] == cls_name].set_index('Horizon').loc[h_order]
            for bar, (_, row) in zip(ax.containers[c_i], c_sub.iterrows()):
                pct = row['Persen']
                ax.text(bar.get_x() + bar.get_width() / 2.0, bar.get_height() * 1.18,
                        f'{pct:.2f}%' if pct >= 1.0 else f'{pct:.3f}%', ha='center', va='bottom',
                        fontsize=8, color='#2d3748')
        ov_pos = sub[(sub['Horizon'] == 'Keseluruhan') & (sub['Kelas'] == 'Terbakar (1)')]['Persen'].iloc[0]
        ax.set_title(f"{t_label} — Rasio Ketidakseimbangan (≈{(100.0 - ov_pos) / max(ov_pos, 1e-9):.0f}:1)", fontweight='bold')
        ax.legend(loc='upper center', ncol=2, frameon=True, facecolor='white', edgecolor='#e2e8f0')
    axes[1].set_xlabel('Horizon Prediksi')

In [ ]:
plot_target_imbalance_2x1(eda_arrays)

Grafik dua panel dengan skala vertikal logaritmik tersebut memperlihatkan ketimpangan kelas yang sangat ekstrem pada kedua jenis target prediksi di seluruh horizon waktu. Pada target piksel terbakar baru di panel atas rasio ketidakseimbangan mendekati 244 banding 1 karena piksel terbakar hanya mencakup 0,409 persen secara keseluruhan dari 0,233 persen pada t+1 hingga 0,579 persen pada t+3, sementara pada target area terbakar kumulatif di panel bawah rasio ketimpangan tercatat sekitar 29 banding 1 dengan kelas positif sebesar 3,29 persen, sehingga rancangan pemodelan membutuhkan fungsi kerugian penekan ketimpangan kelas serta evaluasi berbasis presisi dan daya ingat.

#### Multi-Horizon Wildfire Spread Progression and Target Masking

In [ ]:
# Menggambarkan evolusi spasial penjalaran api selama 72 jam pada sampel terpilih data latih dan data validasi.
def plot_progression_samples(df, raw_arrays):
    chosen = []
    for sp in ['Data Latih', 'Data Validasi']:
        max_init, min_val = (48, 0.85) if sp == 'Data Latih' else (90, 0.70)
        sub = df[(df['split'] == sp) & (df['init_fire_px'] >= 5) & (df['init_fire_px'] <= max_init)
                 & (df['y_h3_px'] > df['y_h2_px']) & (df['y_h2_px'] > df['y_h1_px']) & (df['y_h1_px'] > df['init_fire_px'])
                 & (df['yn_h3_px'] >= 60) & (df['valid_h3_frac'] >= min_val)].copy()
        if sub.empty:
            sub = df[df['split'] == sp].copy()
        if sp == 'Data Validasi' and chosen:
            sub_diff = sub[sub['province'] != chosen[0]['province']]
            if len(sub_diff) > 0:
                sub = sub_diff
        chosen.append(sub.sort_values(['yn_h3_px', 'valid_h3_frac'], ascending=[False, False]).iloc[0].to_dict())
    def _smooth_mask(mask_2d, sigma=1.3):
        return ndi.gaussian_filter(ndi.zoom(mask_2d.astype(np.float32), 4, order=1), sigma=sigma) >= 0.45
    fig, axes = plt.subplots(2, 4, figsize=(16.5, 8.8))
    stage_titles = ['Hari t (Api Awal)', 'Hari t+1 (24 jam)', 'Hari t+2 (48 jam)', 'Hari t+3 (72 jam)']
    for row_i, s_meta in enumerate(chosen):
        arr = raw_arrays[s_meta['split']]
        idx = int(s_meta['idx'])
        bb = arr['bbox'][idx]
        extent = [float(bb[0]), float(bb[2]), float(bb[1]), float(bb[3])]
        st, y, yn, val_m = arr['state_t'][idx], arr['y'][idx], arr['y_new'][idx], arr['valid'][idx] > 0
        for col_i in range(4):
            ax = axes[row_i, col_i]
            ax.set_xlim(extent[0], extent[1])
            ax.set_ylim(extent[2], extent[3])
            ax.set_aspect('equal')
            add_esri_basemap(ax, extent, attribution_size=6 if (row_i == 1 and col_i == 0) else 0)
            rgba = np.zeros((256, 256, 4), dtype=np.float32)
            v_mask = val_m[0 if col_i == 0 else col_i - 1]
            rgba[_smooth_mask(~v_mask, sigma=1.1)] = [0.39, 0.45, 0.55, 0.40]
            if col_i == 0:
                active_t = (st[1] > 0.5) & v_mask
                rgba[_smooth_mask((st[2] > 0.5) & v_mask)] = [0.45, 0.22, 0.16, 0.65]
                rgba[_smooth_mask(active_t, sigma=1.2)] = [0.79, 0.64, 0.15, 0.95]
                px_txt = f'Api Aktif = {int(active_t.sum())} px'
            else:
                h_i = col_i - 1
                cum_y, new_y = (y[h_i] > 0) & v_mask, (yn[h_i] > 0) & v_mask
                rgba[_smooth_mask(cum_y & ~new_y)] = [0.55, 0.15, 0.15, 0.70]
                rgba[_smooth_mask(new_y)] = [0.88, 0.48, 0.25, 0.92]
                px_txt = f'Kum = {int(cum_y.sum())*375/10000:.2f} ha | Baru = {int(new_y.sum())*375/10000:.2f} ha'
            ax.imshow(rgba, extent=extent, origin='upper', interpolation='bilinear', zorder=5)
            add_lonlat(ax)
            ax.xaxis.set_major_locator(MaxNLocator(3))
            ax.yaxis.set_major_locator(MaxNLocator(4))
            if col_i == 0:
                add_north_arrow(ax, x=0.90, y=0.89)
                add_scalebar(ax, x=0.05, y=0.06)
                ax.set_ylabel(f"Sampel {row_i + 1}: {s_meta['province']}\n({s_meta['date'].strftime('%d %b %Y')})", fontsize=9.5, fontweight='bold')
            else:
                ax.tick_params(labelleft=False)
            if row_i == 0:
                ax.tick_params(labelbottom=False)
            ax.set_title(f'{stage_titles[col_i]}\n{px_txt}', fontsize=8, fontweight='bold')
    legend_handles = [
        Patch(facecolor=(0.79, 0.64, 0.15, 0.95), edgecolor='black', label='Api Aktif Awal (t)'),
        Patch(facecolor=(0.55, 0.15, 0.15, 0.72), edgecolor='black', label='Area Terbakar Kumulatif (y)'),
        Patch(facecolor=(0.88, 0.48, 0.25, 0.92), edgecolor='black', label='Perluasan Terbakar Baru (y_new)'),
        Patch(facecolor=(0.39, 0.45, 0.55, 0.45), edgecolor='black', label='Area Termasking (Awan/Air)'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=4, frameon=True, facecolor='white', bbox_to_anchor=(0.5, -0.02))
    fig.suptitle('Visualisasi Perkembangan Perluasan Area Kebakaran dalam 72 Jam', fontsize=12.5, fontweight='bold', y=1.01)
    plt.tight_layout(rect=[0, 0.03, 1, 0.98], h_pad=3)

In [ ]:
plot_progression_samples(eda_df, eda_arrays)

Peta perkembangan spasial di atas memperlihatkan perluasan area kebakaran selama 72 jam pada dua sampel nyata di Kalimantan Tengah dan Kalimantan Selatan beserta wilayah yang tertutup awan atau badan air. Pada sampel pertama tanggal 3 November 2019 perluasan api baru berkembang bertahap dari 2,14 hektar pada t+1 menjadi 4,16 hektar pada t+2 hingga 9,68 hektar pada t+3, sedangkan pada sampel kedua tanggal 10 September 2019 area terbakar baru melonjak dari 1,69 hektar menjadi 13,84 hektar pada t+3 yang secara visual menegaskan peran penghalang awan serta alur sungai terhadap visibilitas pengamatan dan batas penjalaran api.

#### Dominant Land Cover Distribution and Anthropogenic Spread Drivers

In [ ]:
# Memetakan kategori tutupan lahan IGBP dominan pada setiap lokasi titik kejadian kebakaran di Kalimantan.
def plot_landcover_map(df, prov_gdf):
    df = df.copy()
    top_lcs = df['dom_lc'].value_counts().head(6).index.tolist()
    df['Tutupan Lahan'] = df['dom_lc'].apply(
        lambda c: IGBP_LABELS_ID.get(c, f'Kelas {c}') if c in top_lcs else 'Lainnya'
    )
    cats = [IGBP_LABELS_ID.get(c, f'Kelas {c}') for c in top_lcs]
    if 'Lainnya' in df['Tutupan Lahan'].values:
        cats.append('Lainnya')

    markers_list = ['o', 's', '^', 'D', 'v', 'P', 'X']
    fig, ax = plt.subplots(figsize=(7.5, 6.0))
    setup_kalimantan_map(ax, prov_gdf, lw=0.9, attribution_size=6, north_xy=(0.93, 0.92), scale_xy=(0.04, 0.05))
    for i, cat in enumerate(cats):
        sub_c = df[df['Tutupan Lahan'] == cat]
        ax.scatter(
            sub_c['lon'], sub_c['lat'],
            marker=markers_list[i % len(markers_list)],
            color=SCIENTIFIC_PALETTE[i % len(SCIENTIFIC_PALETTE)],
            s=22, alpha=0.78, edgecolors='white',
            linewidths=0.3, label=cat, zorder=5
        )
    ax.set_title('Sebaran Tutupan Lahan Dominan pada Titik Kebakaran', fontweight='bold')
    leg = ax.legend(loc='lower right', fontsize=5, frameon=True, facecolor='white', markerscale=1.1, 
            title='Tutupan Lahan', title_fontsize=6.5)
    for text in leg.get_texts():
        text.set_fontweight('bold')
        leg.get_title().set_fontweight('bold')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_landcover_map(eda_df, prov_gdf)

Peta kategorikal di atas menampilkan jenis tutupan lahan dominan pada setiap lokasi titik kejadian kebakaran di Pulau Kalimantan. Sebaran titik memperlihatkan bahwa tipe Sabana Berkayu dan Hutan Berdaun Lebar Hijau mendominasi sebagian besar kebakaran di Kalimantan Barat, Kalimantan Tengah, Kalimantan Timur, dan Kalimantan Utara, sementara kejadian pada kategori Sabana berkumpul padat di wilayah selatan dan tenggara Kalimantan Selatan.

In [ ]:
palette={'Terdeforestasi': SCIENTIFIC_PALETTE[1], 'Kanopi Utuh': SCIENTIFIC_PALETTE[2]}

In [ ]:
# Menganalisis pengaruh jarak patch ke jalan terdekat dan riwayat deforestasi terhadap rata-rata luas area terbakar baru.
def plot_road_deforestation_analysis(df):
    df = df.copy()
    df['dist_road_km'] = df['dist_road_mean'] / 1000.0
    df['area_ha'] = df['yn_total_px'] * 375 / 10000
    df['Riwayat Lahan'] = np.where(df['floss_frac'] >= 0.5, "Terdeforestasi", 'Kanopi Utuh')
    df['Jarak ke Jalan'] = pd.cut(df['dist_road_km'], bins=[0, 0.5, 1.0, 2.0, 4.0, 8.0, 60.0], labels=['<0,5', '0,5–1', '1–2', '2–4', '4–8', '>8 km'])
    fig, ax = plt.subplots(figsize=(7.5, 5.0))
    sns.barplot(data=df, x='Jarak ke Jalan', y='area_ha', hue='Riwayat Lahan',
        palette={'Terdeforestasi': SCIENTIFIC_PALETTE[3], 'Kanopi Utuh': SCIENTIFIC_PALETTE[1]},
        errorbar=('ci', 95), ax=ax, edgecolor='white')
    ax.set_xlabel('Rata-rata Jarak Patch ke Jalan Terdekat (km)')
    ax.set_ylabel('Rata-rata Area Terbakar Baru (ha)')
    ax.set_title('Faktor Antropogenik terhadap Persebaran Kebakaran', fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ax.legend(title='Riwayat Deforestasi', frameon=True, facecolor='white', fontsize=8, title_fontsize=9, edgecolor='#e2e8f0')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_road_deforestation_analysis(eda_df)

Grafik batang berkelompok tersebut membandingkan rata-rata luas area terbakar baru berdasarkan jarak rata-rata *patch* ke jalan terdekat serta riwayat deforestasi lahan. Lahan terdeforestasi secara konsisten mencatat rata-rata luas terbakar baru yang jauh lebih tinggi dibandingkan kanopi utuh pada jarak menengah hingga jauh dari jalan, khususnya pada rentang jarak 2 hingga 4 kilometer yang mencapai sekitar 2,5 hektar berbanding di bawah 0,5 hektar dan pada jarak lebih dari 8 kilometer yang melampaui 3 hektar karena akumulasi bahan bakar kering di kawasan pedalaman yang sulit dijangkau upaya pemadaman.

#### Polar Distribution of Wind Advection Direction at Active Fire Pixels

In [ ]:
# Menampilkan diagram polar distribusi arah adveksi angin pada piksel api aktif serta bobot penjalarannya.
def plot_wind_polar(raw_arrays):
    thetas, spread_w = [], []
    for d in raw_arrays.values():
        v = d['valid'][:, 0] > 0
        m = (d['state_t'][:, 1] > 0.5) & v
        thetas.append(d['ca_phys'][:, 1][m])
        patch_yn = ((d['y_new'][:, 0] > 0) & v).sum(axis=(1, 2))
        spread_w.append(np.broadcast_to(patch_yn[:, None, None], m.shape)[m])
    th_all = np.concatenate(thetas) % (2 * np.pi)
    wt_all = np.concatenate(spread_w) + 1.0
    fig = plt.figure(figsize=(6.4, 6.0))
    ax = fig.add_subplot(1, 1, 1, projection='polar')
    n_bins = 24
    edges = np.linspace(0, 2 * np.pi, n_bins + 1)
    hist_cnt, _ = np.histogram(th_all, bins=edges, density=True)
    hist_wt, _ = np.histogram(th_all, bins=edges, weights=wt_all, density=True)
    centers, width = (edges[:-1] + edges[1:]) / 2.0, 2 * np.pi / n_bins * 0.88
    ax.bar(centers, hist_cnt, width=width, color=SCIENTIFIC_PALETTE[0], alpha=0.45, edgecolor=SCIENTIFIC_PALETTE[0], label='Frekuensi Api Aktif')
    ax.bar(centers, hist_wt, width=width * 0.62, color=SCIENTIFIC_PALETTE[1], alpha=0.82, edgecolor='#9c4221', label='Dibobotkan Penjalaran (y_new)')
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(1)
    ax.set_title('Distribusi Arah Adveksi Angin pada Titik Api Aktif', fontweight='bold', pad=15)
    ax.legend(loc='upper right', bbox_to_anchor=(1.22, 1.08), fontsize=8, frameon=True, facecolor='white')

In [ ]:
plot_wind_polar(eda_arrays)

Diagram polar di atas memperlihatkan distribusi frekuensi arah adveksi angin pada piksel api aktif bersama distribusi yang dibobotkan oleh luas penjalaran api baru. Sebagian besar piksel api aktif berkumpul pada sudut arah angin sekitar 90 hingga 135 derajat, dan ketika dibobotkan oleh luas area terbakar baru puncaknya bergeser lebih mengerucut mendekati 90 hingga 110 derajat yang membuktikan keterkaitan antara arah angin dan dominasi arah penjalaran front api.

#### Global Climate Teleconnection (ONI vs. DMI) and Wildfire Spread Regimes

In [ ]:
# Memetakan hubungan antara fase indeks iklim global ONI dan DMI dengan rata-rata luas persebaran api.
def plot_enso_climate_regimes(df: pd.DataFrame) -> None:
    agg_df = (
        df.assign(**{
            "Fase ENSO": np.select([df["oni"] >= 0.5, df["oni"] <= -0.5], ["El Niño", "La Niña"], "Netral"),
            "yn_ha": df["yn_total_px"] * 375 / 10000,
        }).groupby(["oni", "dmi", "Fase ENSO"], as_index=False).agg(**{"Rata-rata Persebaran (ha)": ("yn_ha", "mean")}))
    fig, ax = plt.subplots(figsize=(8.6, 5.8))
    palette = {"El Niño": SCIENTIFIC_PALETTE[1], "Netral": SCIENTIFIC_PALETTE[5], "La Niña": SCIENTIFIC_PALETTE[0]}
    x_min, x_max = min(-1.15, agg_df["oni"].min() - 0.15), max(1.95, agg_df["oni"].max() + 0.15)
    y_min, y_max = min(-0.55, agg_df["dmi"].min() - 0.10), max(1.08, agg_df["dmi"].max() + 0.12)
    ax.set(xlim=(x_min, x_max), ylim=(y_min, y_max),
           xlabel="Indeks Osilasi El Niño (ONI)", ylabel="Indeks Mode Dipol Samudra Hindia (DMI)")
    ax.set_title("Pengaruh Telekoneksi Iklim Global terhadap Besarnya Persebaran Kebakaran", fontweight="bold")
    ax.grid(True, ls="--", alpha=0.3)

    for bound, span, lbl, t_col, p_key in [(0.5, x_max, "El Niño", "#9c4221", "El Niño"), (-0.5, x_min, "La Niña", palette["La Niña"], "La Niña")]:
        ax.axvspan(bound, span, color=palette[p_key], alpha=0.09, zorder=0)
        ax.axvline(bound, color=palette[p_key], ls="--", lw=1.1, alpha=0.85)
        ax.text((bound + span) / 2, y_max - 0.08, lbl, fontsize=9.5, fontweight="bold", color=t_col,
                ha="center", va="top", bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=t_col, alpha=0.9), zorder=4)
    ax.axhline(0.4, color=SCIENTIFIC_PALETTE[4], ls=":", lw=1.2, alpha=0.9)

    sns.scatterplot(data=agg_df, x="oni", y="dmi", hue="Fase ENSO", hue_order=list(palette),
                    size="Rata-rata Persebaran (ha)", sizes=(35, 420), alpha=0.82, palette=palette,
                    ax=ax, edgecolor="white", lw=0.8, zorder=3, legend=False)

    leg_cfg = dict(loc="lower right", fontsize=7, title_fontsize=8, labelspacing=0.6, borderpad=0.7, frameon=True, facecolor="white", edgecolor="#e2e8f0")
    h_hue = [plt.Line2D([], [], marker="o", ls="none", ms=6, mfc=c, mec="white", label=k) for k, c in palette.items()]
    h_size = [plt.scatter([], [], s=s, color="#808080", alpha=0.65, ec="white", label=l) for s, l in [(25, "Rendah"), (90, "Sedang"), (180, "Tinggi")]]
    ax.add_artist(ax.legend(handles=h_hue, title="Fase ENSO", bbox_to_anchor=(0.85, 0.04), **leg_cfg))
    ax.legend(handles=h_size, title="Persebaran Api", bbox_to_anchor=(1.0, 0.04), handletextpad=0.8, **leg_cfg)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_enso_climate_regimes(eda_df)

Grafik sebar gelembung di atas memetakan posisi peristiwa kebakaran pada bidang indeks iklim ONI dan DMI dengan ukuran lingkaran mencerminkan besarnya penjalaran api. Fase El Nino pada kuadran kanan atas ketika nilai ONI minimal 0,5 dan DMI melebihi 0,4 memperlihatkan banyak kejadian kebakaran berukuran sedang hingga besar akibat sinergi kekeringan dua samudra, meskipun pada kondisi Netral dan La Nina sesekali masih terjadi kebakaran luas yang dipicu oleh kondisi bahan bakar maupun faktor lokal.

#### Provincial Wildfire Spread Comparison: Peatland vs. Non-Peatland

In [ ]:
# Membandingkan rata-rata luas piksel terbakar baru pada horizon 72 jam antara lahan gambut dan lahan nongambut di tiap provinsi.
def plot_peatland_mineral_province(df):
    df = df.copy()
    df['yn_h3_px'] = df['yn_h3_px'] * 375 / 10000
    df['Provinsi'] = df['province'].str.replace('Kalimantan ', 'Kal. ')
    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    sns.barplot(data=df, x='Provinsi', y='yn_h3_px', hue='is_peat_dom',
                palette={'Lahan Gambut': SCIENTIFIC_PALETTE[3], 'Lahan Non-Gambut': SCIENTIFIC_PALETTE[1]},
                errorbar=('ci', 95), ax=ax, edgecolor='white')
    ax.set_xlabel('Provinsi', fontsize=9)
    ax.set_ylabel('Rata-rata Luas Piksel Terbakar Baru pada t+3 (ha)', fontsize=9)
    ax.set_title('Laju Persebaran Api antara Lahan Gambut dan Non-Gambut per Provinsi', fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ax.legend(title='Lahan Dominan', frameon=True, facecolor='white', fontsize=8, title_fontsize=9, edgecolor='#e2e8f0')

In [ ]:
plot_peatland_mineral_province(eda_df)

Grafik batang berkelompok di atas membandingkan rata-rata luas piksel terbakar baru pada horizon 72 jam antara lahan gambut dominan dan lahan nongambut di lima provinsi Kalimantan. Lahan gambut menghasilkan rata-rata area terbakar jauh lebih luas dibandingkan lahan nongambut di Kalimantan Selatan sekitar 1,02 berbanding 0,54 hektar, Kalimantan Tengah sekitar 0,74 berbanding 0,23 hektar, dan Kalimantan Barat sekitar 0,55 berbanding 0,11 hektar, sedangkan di Kalimantan Timur kebakaran rata-rata lebih besar pada lahan nongambut sekitar 0,81 berbanding 0,15 hektar dan di Kalimantan Utara kejadian teramati pada lahan nongambut.

# **4. Data Preprocessing**

Bagian ini mengonstruksi masukan siap latih beserta masker domain spasial agar proses pembelajaran model berfokus pada area pengamatan yang sah secara fisik. Penyesuaian augmentasi geometris turut diselaraskan dengan hukum arah vektor angin sehingga informasi meteorologi tidak menyimpang.

* Membentuk domain evaluasi berupa zona penyangga di sekeliling titik api awal dengan menyisihkan wilayah yang telah hangus maupun area tertutup awan dan air.
* Menerapkan augmentasi pembalikan horizontal dan vertikal secara acak yang ikut membalik komponen tanda vektor kecepatan angin.
* Menyusun kelas dataset PyTorch dan *sampler* berbobot untuk mengatasi kelangkaan sampel yang memiliki perluasan api aktif.

### **Masking Fire Event**

Subbagian ini mengimplementasikan fungsi `make_eval_domain` yang membentuk masker evaluasi melalui dilasi morfologis beradius 4, 6, dan 8 piksel secara berurutan untuk horizon t+1, t+2, dan t+3 di sekitar posisi api awal dengan menyisihkan area terbakar sebelumnya serta piksel tidak valid, sehingga mampu merangkum cakupan target sebesar 84,13 persen pada t+1 dan 89,48 persen pada t+3.

In [ ]:
# Membentuk masker domain evaluasi per horizon melalui dilasi spasial berulang yang mengecualikan area awal dan piksel tidak valid.
def make_eval_domain(valid, burned_t, fire_t, fire_prev, radii=BUFFER_RADII):
    valid_b = np.asarray(valid) > 0.5
    N, H_dim, W_dim = valid_b.shape[0], valid_b.shape[-2], valid_b.shape[-1]
    dom = np.zeros((N, H, H_dim, W_dim), dtype=bool)
    foot = ndi.generate_binary_structure(2, 2)
    for i in range(N):
        curr = fire_t[i] | fire_prev[i]
        b_mask = burned_t[i]
        for h, r in enumerate(radii):
            v_mask = valid_b[i, h] if valid_b.ndim == 4 else valid_b[i]
            dil = ndi.binary_dilation(curr, structure=foot, iterations=r)
            dom[i, h] = dil & (~b_mask) & v_mask
    return dom

In [ ]:
# Menerapkan pembentukan masker domain evaluasi secara seragam pada himpunan data latih dan data validasi.
for d in (train_d, val_d):
    d['dom'] = make_eval_domain(d['valid'], d['burned_t'], d['x'][:, 0] > 0.5, d['x'][:, 1] > 0.5)

In [ ]:
assert not (train_d['dom'] & train_d['burned_t'][:, None]).any()
cov_t1 = (train_d['y_new'][:, 0] & train_d['dom'][:, 0]).sum() / max(1, train_d['y_new'][:, 0].sum()) * 100
cov_t3 = (train_d['y_new'][:, 2] & train_d['dom'][:, 2]).sum() / max(1, train_d['y_new'][:, 2].sum()) * 100
print(f'Domain recall ceiling: t+1 = {cov_t1:.2f}%, t+3 = {cov_t3:.2f}%')

### **Augmentations**

Subbagian ini mendefinisikan fungsi `apply_flip_aug` untuk melakukan pembalikan citra dan masker secara acak pada sumbu horizontal maupun vertikal dengan peluang 50 persen, sembari membalik tanda komponen angin zonal saat pembalikan horizontal dan komponen meridional saat pembalikan vertikal agar arah adveksi tetap akurat secara fisika.

In [ ]:
# Menerapkan augmentasi mirror horizontal dan vertikal disertai pembalikan tanda vektor arah angin serta kemiringan lereng.
def apply_flip_aug(x, y, dom, state, phys, fg):
    if random.random() < 0.5:
        x, y, dom, state = torch.flip(x, [-1]), torch.flip(y, [-1]), torch.flip(dom, [-1]), torch.flip(state, [-1])
        phys, fg = torch.flip(phys, [-1]), torch.flip(fg, [-1])
        x[26], x[30], fg[:, 0] = -x[26], -x[30], -fg[:, 0]
        phys[1], phys[3] = (math.pi - phys[1]) % (2 * math.pi), (math.pi - phys[3]) % (2 * math.pi)
    if random.random() < 0.5:
        x, y, dom, state = torch.flip(x, [-2]), torch.flip(y, [-2]), torch.flip(dom, [-2]), torch.flip(state, [-2])
        phys, fg = torch.flip(phys, [-2]), torch.flip(fg, [-2])
        x[27], x[31], fg[:, 1] = -x[27], -x[31], -fg[:, 1]
        phys[1], phys[3] = (-phys[1]) % (2 * math.pi), (-phys[3]) % (2 * math.pi)
    return x, y, dom, state, phys, fg

### **Create Custom Dataset**

Subbagian ini membungkus tensor masukan dan target ke dalam kelas `FireDataset` serta mengonfigurasi `WeightedRandomSampler` dengan bobot 1,0 bagi sampel yang memiliki perluasan api dan 0,1 bagi sampel tanpa api baru agar *DataLoader* lebih sering menyajikan contoh penjalaran aktif selama pelatihan.

In [ ]:
# Membungkus susunan fitur, target perluasan api, domain evaluasi, status awal, dan variabel pendorong fisika menjadi PyTorch Dataset.
class FireDataset(Dataset):
    def __init__(self, d, augment=False):
        self.x, self.y = d['x'], d['y_new'].astype(np.float32)
        self.dom, self.state_t = d['dom'], d['state_t']
        self.ca_phys, self.forcing_g = d['ca_phys'], d['forcing_g']
        self.augment = augment
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        x, y = torch.from_numpy(self.x[idx].copy()), torch.from_numpy(self.y[idx].copy())
        dom, state = torch.from_numpy(self.dom[idx].copy()), torch.from_numpy(self.state_t[idx].copy())
        phys, fg = torch.from_numpy(self.ca_phys[idx].copy()), torch.from_numpy(self.forcing_g[idx].copy())
        if self.augment:
            x, y, dom, state, phys, fg = apply_flip_aug(x, y, dom, state, phys, fg)
        return x, y, dom, state, phys, fg 

In [ ]:
# Menyeimbangkan komposisi batch pelatihan dengan memberi bobot pengambilan sepuluh kali lipat lebih tinggi bagi sampel berapi baru.
act_cnt = train_d['y_new'].sum((1, 2, 3))
weights = np.where(act_cnt > 0, 1.0, 0.1)
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
_pin = torch.cuda.is_available()
_dl_kw = {'pin_memory': _pin, 'num_workers': 0}
train_loader = DataLoader(FireDataset(train_d, augment=True), batch_size=BATCH_SIZE, sampler=sampler, **_dl_kw)
val_loader = DataLoader(FireDataset(val_d, augment=False), batch_size=BATCH_SIZE, shuffle=False, **_dl_kw)

# **5. Training Functions**

Bagian ini membangun seluruh perangkat lunak inti untuk melatih dan mengevaluasi jaringan saraf tiruan secara stabil dan efisien. Mekanisme optimasi dirancang agar tahan terhadap ketimpangan kelas ekstrem, mendukung presisi campuran FP16, serta mampu mengelola penyimpanan bobot pada lingkungan satu maupun dua GPU.

* Merumuskan fungsi kerugian komposit `physics_loss` yang menggabungkan kesesuaian bentuk, penekanan tepi api, dan kalibrasi luas area.
* Mengimplementasikan fungsi pelatihan satu *epoch* dengan *GradScaler* presisi campuran dan pemotongan norma gradien.
* Membangun fungsi evaluasi validasi tanpa pelacakan gradien guna memilih bobot model terbaik di setiap akhir *epoch*.
* Merangkai alur `fit_model` yang mengatur penjadwalan *learning rate* per arsitektur, pembaruan log sebaris, penyimpanan *checkpoint*, serta pembersihan memori GPU.

### **Composite Loss**

Subbagian ini mendefinisikan fungsi kerugian gabungan `physics_loss` yang memadukan *Focal Tversky Loss* dengan parameter $\alpha = 0{,}75$ guna menekan kegagalan deteksi piksel terbakar, *Frontier Weighted Binary Cross Entropy* dengan peta pembobot batas $w_{\text{map}} = (1 + 9 \cdot \text{frontier}) \cdot m$ untuk mempertajam prediksi pada garis tepi penjalaran, serta penalti galat luas absolut ternormalisasi guna menjaga proporsi luasan terbakar.

$$\mathcal{L}_{\text{total}} = w_{\text{ft}} \mathcal{L}_{\text{FT}} + w_{\text{bce}} \mathcal{L}_{\text{FBCE}} + w_{\text{area}} \mathcal{L}_{\text{area}}$$

$$\mathcal{L}_{\text{FT}} = \frac{1}{H} \sum_{h=1}^{H} \left(1 - \frac{\text{TP}_h + 1}{\text{TP}_h + 0{,}75\,\text{FN}_h + 0{,}25\,\text{FP}_h + 1}\right)^{0{,}75}$$

$$\mathcal{L}_{\text{area}} = \frac{1}{BH} \sum_{b=1}^{B} \sum_{h=1}^{H} \frac{\left|\sum_{i,j} p_{b,h,i,j} - \sum_{i,j} t_{b,h,i,j}\right|}{\sum_{i,j} t_{b,h,i,j} + 10}$$

In [ ]:
# Menghitung loss gabungan yang memadukan Focal Tversky, pembobotan garis tepi api, dan penalti kesalahan total luas.
def physics_loss(probs, targets, dom, alpha=0.75, smooth=1.0, w_ft=1.0, w_bce=0.5, w_area=0.25):
    m = dom.float()
    p = probs.clamp(1e-6, 1.0 - 1e-6) * m
    t = targets.float() * m
    tp = (p * t).sum((0, 2, 3))
    fn = ((1.0 - p) * t).sum((0, 2, 3))
    fp = (p * (1.0 - t)).sum((0, 2, 3))
    ti = (tp + smooth) / (tp + alpha * fn + (1.0 - alpha) * fp + smooth)
    l_ft = ((1.0 - ti) ** 0.75).mean()
    if w_bce == 0.0 and w_area == 0.0:
        return w_ft * l_ft
    # Memberikan bobot pembobotan sepuluh kali lipat pada piksel perbatasan garis penjalaran api dalam perhitungan entropi silang.
    with torch.no_grad():
        t_dil = F.max_pool2d(t, kernel_size=3, stride=1, padding=1)
        frontier = ((t_dil - t) > 0.5).float()
        w_map = (1.0 + 9.0 * frontier) * m
    bce_raw = -(t * torch.log(p.clamp(min=1e-6)) + (1.0 - t) * torch.log((1.0 - p).clamp(min=1e-6)))
    l_bce = (w_map * bce_raw).sum() / w_map.sum().clamp(min=1.0)
    l_area = ((p.sum((2, 3)) - t.sum((2, 3))).abs() / (t.sum((2, 3)) + 10.0)).mean()
    return w_ft * l_ft + w_bce * l_bce + w_area * l_area

### **Training Loop**

Subbagian ini menyusun fungsi `train_epoch` yang mengirimkan tensor *batch* secara nonblokir ke GPU, menjalankan inferensi maju dalam presisi campuran FP16 serta menghitung nilai `physics_loss` pada presisi penuh FP32, lalu melakukan perambatan balik dengan pembatasan norma gradien maksimal bernilai 1 dan memajukan jadwal *learning rate* pada akhir *epoch*.

In [ ]:
# Menjalankan satu epoch pelatihan model memakai presisi FP16.
def train_epoch(model, loader, optimizer, scheduler=None, loss_fn=physics_loss, scaler=None):
    model.train()
    losses = []
    use_amp = scaler is not None and scaler.is_enabled()
    for x_b, y_b, dom_b, st_b, ph_b, fg_b in loader:
        x_b, y_b, dom_b = x_b.to(DEVICE, non_blocking=True), y_b.to(DEVICE, non_blocking=True), dom_b.to(DEVICE, non_blocking=True)
        st_b, ph_b, fg_b = st_b.to(DEVICE, non_blocking=True), ph_b.to(DEVICE, non_blocking=True), fg_b.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        ctx = {'state_t': st_b, 'ca_phys': ph_b, 'forcing_g': fg_b}
        with torch.amp.autocast('cuda', enabled=use_amp):
            probs = model(x_b, ctx)
        loss = loss_fn(probs.float(), y_b, dom_b)
        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        losses.append(loss.item())
    if scheduler is not None:
        scheduler.step()
    return float(np.mean(losses))

### **Validation Loop**

Subbagian ini mengimplementasikan fungsi `eval_epoch` dalam blok `@torch.no_grad()` untuk menghitung rata-rata kerugian validasi pada seluruh *batch* data validasi dengan presisi campuran pada tahap maju dan kerugian FP32 sebagai dasar pemilihan model terbaik.

In [ ]:
@torch.no_grad()
# Mengevaluasi rata-rata nilai loss model pada seluruh batch validasi.
def eval_epoch(model, loader, loss_fn=physics_loss):
    model.eval()
    losses = []
    use_amp = torch.cuda.is_available()
    for x_b, y_b, dom_b, st_b, ph_b, fg_b in loader:
        x_b, y_b, dom_b = x_b.to(DEVICE, non_blocking=True), y_b.to(DEVICE, non_blocking=True), dom_b.to(DEVICE, non_blocking=True)
        st_b, ph_b, fg_b = st_b.to(DEVICE, non_blocking=True), ph_b.to(DEVICE, non_blocking=True), fg_b.to(DEVICE, non_blocking=True)
        ctx = {'state_t': st_b, 'ca_phys': ph_b, 'forcing_g': fg_b}
        with torch.amp.autocast('cuda', enabled=use_amp):
            probs = model(x_b, ctx)
        loss = loss_fn(probs.float(), y_b, dom_b)
        losses.append(loss.item())
    return float(np.mean(losses))

### **Training Pipeline**

Subbagian ini mengintegrasikan pelatihan lengkap ke dalam fungsi `fit_model` yang mendukung pembungkusan *DataParallel* pada multi GPU, penyesuaian *learning rate* dengan optimizer SOAP serta *warmup* dan *cosine annealing* yang lebih agresif bagi DeepLabV3+ dengan CA dibandingkan U-Net, penyimpanan berkas bobot terbaik serta status lanjutan setiap lima *epoch*, hingga pembersihan memori setelah pelatihan usai.

In [ ]:
# Pipeline siklus pelatihan penuh meliputi dukungan multi GPU, penjadwalan optimasi SOAP, dan penyimpanan checkpoint model.
def fit_model(name, model, train_ld, val_ld, epochs=EPOCHS, retrain=RETRAIN, loss_fn=physics_loss):
    if USE_MULTI_GPU and not isinstance(model, nn.DataParallel):
        model = nn.DataParallel(model)
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    safe_name = name.replace('+', 'plus').replace('-', '_').replace('/', '_').replace(' ', '_')
    best_ckpt_path = CKPT_DIR / f'{safe_name}_best.pt'
    last_ckpt_path = CKPT_DIR / f'{safe_name}_last.pt'
    is_dl_ca = isinstance(raw_model, FireCA) and isinstance(raw_model.backbone, DeepLabHighResBackbone)
    is_dl = is_dl_ca or ('DeepLab' in name)
    is_unet = 'U-Net' in name
    # Menyesuaikan nilai learning rate, peluruhan bobot, serta panjang fase pemanasan berdasarkan karakteristik konvergensi masing-masing arsitektur.
    if is_dl_ca:
        lr_use, wd_use, wu_ep, wu_start = LR * 2.5, 3e-4, min(3, max(1, epochs // 3)), 0.30
    elif is_dl:
        lr_use, wd_use, wu_ep, wu_start = LR * 1.5, 5e-4, min(4, max(1, epochs // 3)), 0.15
    elif is_unet:
        lr_use, wd_use, wu_ep, wu_start = LR * 0.5, 1e-3, min(10, max(1, epochs // 2)), 0.05
    else:
        lr_use, wd_use, wu_ep, wu_start = LR, 1e-3, min(5, max(1, epochs // 2)), 0.05
    opt = SOAP(model.parameters(), lr=lr_use, weight_decay=wd_use, precondition_frequency=10)
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
    warmup_sched = torch.optim.lr_scheduler.LinearLR(opt, start_factor=wu_start, total_iters=wu_ep)
    if is_dl:
        cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs - wu_ep), eta_min=1e-6)
    else:
        cosine_sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=15, T_mult=2, eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[warmup_sched, cosine_sched], milestones=[wu_ep])
    start_ep, best_loss, history = 1, float('inf'), {'train_loss': [], 'val_loss': [], 'lr': []}
    if not retrain and best_ckpt_path.exists() and last_ckpt_path.exists():
        ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)
        if ckpt.get('epoch', 0) >= epochs:
            raw_model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE, weights_only=True))
            print(f'[{name}] Checkpoint dimuat (epoch {ckpt.get("epoch", 0)}/{epochs})')
            return model, ckpt.get('history', history)
        raw_model.load_state_dict(ckpt['model_state_dict'])
        opt.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_ep, best_loss, history = ckpt['epoch'] + 1, ckpt['best_loss'], ckpt['history']
    if start_ep <= epochs:
        print(f'\r[{name}] Memulai epoch {start_ep}/{epochs}...', end='', flush=True)
    for ep in range(start_ep, epochs + 1):
        tr_l = train_epoch(model, train_ld, opt, scheduler=scheduler, loss_fn=loss_fn, scaler=scaler)
        va_l = eval_epoch(model, val_ld, loss_fn=loss_fn)
        curr_lr = opt.param_groups[0]['lr']
        history['train_loss'].append(tr_l)
        history['val_loss'].append(va_l)
        history['lr'].append(curr_lr)
        if va_l < best_loss:
            best_loss = va_l
            torch.save(raw_model.state_dict(), best_ckpt_path)
        if ep % 5 == 0 or ep == epochs:
            torch.save({
                'epoch': ep,
                'model_state_dict': raw_model.state_dict(),
                'optimizer_state_dict': opt.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_loss': best_loss,
                'history': history,
            }, last_ckpt_path)
        end_ch = '\n' if (ep % 10 == 0 or ep == epochs) else ''
        print(f'\r[{name}] Ep {ep:3d}/{epochs} | tr={tr_l:.4f} | va={va_l:.4f} | best={best_loss:.4f} | lr={curr_lr:.2e}   ', end=end_ch, flush=True)
    # Membersihkan objek optimasi serta memori cache CUDA.
    if best_ckpt_path.exists():
        raw_model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE, weights_only=True))
    del opt, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model, history

# **6. Modelling**

Bagian ini membangun dan melatih seluruh model prediksi penyebaran kebakaran hutan mulai dari pendekatan pembelajaran mesin tabular hingga jaringan segmentasi spasial yang dipadukan dengan persamaan dinamika *Neural Cellular Automata*. Seluruh model dilatih pada himpunan data yang identik lalu dipantau kurva konvergensi kerugiannya serta dicadangkan ke penyimpanan Google Drive.

* Menyelaraskan fitur bahan bakar kontinu serta fitur pendorong fisika atmosfer dan topografi sebagai masukan model.
* Mengonstruksi modul `FireCA` delapan arah berbasis fisika yang mensimulasikan penjalaran api harian dalam presisi FP32 beserta jalur residu loncatan bara.
* Mendefinisikan empat arsitektur *deep learning* yaitu U-Net, DeepLabV3+, U-Net dengan CA, dan DeepLabV3+ dengan CA disertai penguatan resolusi tinggi, serta model pembanding XGBoost.
* Melatih seluruh model secara berurutan satu per satu demi efisiensi VRAM, mengekspor seluruh *checkpoint* ke Drive, dan memplot kurva kerugian latih maupun validasi.

### **Physics-Informed Feature Alignment**

Subbagian ini menyusun modul `ContinuousFuelStem` yang memproyeksikan fitur lahan gambut, tutupan lahan, kelas kelapa sawit, serta masker bahan bakar menjadi empat kanal representasi kontinu, kemudian menggabungkannya lewat `build_physics_features` dan pembungkus `PhysicsDirectWrap` bersama vektor angin, lereng, status kebakaran awal, dan prakiraan cuaca berurutan menjadi 58 kanal masukan untuk model non CA.

In [ ]:
# Memproyeksikan fitur gambut, tutupan lahan, kelapa sawit, dan ketersediaan bahan bakar menjadi empat kanal representasi kontinu.
class ContinuousFuelStem(nn.Module):
    def __init__(self, out_dim=4):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(4, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(8, out_dim, kernel_size=1)
        )
    def forward(self, x, m_fuel):
        fuel_inputs = torch.cat([x[:, 6:9], m_fuel], dim=1)
        return self.proj(fuel_inputs)
    
# Menggabungkan fitur lingkungan statis, status awal, vektor trigonometri angin dan lereng, serta prakiraan cuaca menjadi masukan 58 kanal.
def build_physics_features(x, ctx, fuel_emb):
    v, th_w, slope, th_up, m_fuel = ctx['ca_phys'].chunk(5, 1)
    phys_vec = torch.cat([v, torch.cos(th_w), torch.sin(th_w), slope, torch.cos(th_up), torch.sin(th_up), m_fuel], dim=1)
    fg_flat = ctx['forcing_g'].flatten(1, 2)
    return torch.cat([x, ctx['state_t'], phys_vec, fg_flat, fuel_emb], dim=1)

In [ ]:
# Jaringan segmentasi standar untuk memprediksi probabilitas penyebaran tiga horizon.
class PhysicsDirectWrap(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.fuel_stem = ContinuousFuelStem(out_dim=4)
        self.net = net
    def forward(self, x, ctx):
        m_fuel = ctx['ca_phys'][:, 4:5]
        m_gate = (m_fuel > 0.01).float()
        f_emb = self.fuel_stem(x, m_fuel)
        inp = build_physics_features(x, ctx, f_emb)
        logits = self.net(inp)
        probs = torch.sigmoid(logits) * m_gate
        return probs.clamp(1e-6, 1.0 - 1e-6)

### **Neural Cellular Automata**

Subbagian ini merumuskan modul `FireCA` bersama operasi pergeseran delapan arah `shift8` dan fungsi `compute_phi_and_gamma` yang menghitung laju penyalaan berdasarkan kecepatan angin, arah angin, kemiringan lereng, dan bahan bakar dalam presisi FP32 selama delapan langkah mikro per horizon serta memanfaatkan kanal ketujuh sebagai residu loncatan bara.

$$\Delta I_x = S_x \left(1 - \exp\!\left(-\gamma_x \sum_{u \in \mathcal{N}_8(x)} \phi_{u \to x}\, I_u\right)\right) m_{\text{gate}}$$

In [ ]:
# Mendefinisikan delapan offset ketetanggaan Moore dan fungsi pergeseran tensor spasial bagi kalkulasi penjalaran lokal.
DIRS = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
def shift8(b, pad_mode='constant'):
    hh, ww = b.shape[-2:]
    if pad_mode == 'replicate':
        p = F.pad(b, (1, 1, 1, 1), mode='replicate')
    else:
        p = F.pad(b, (1, 1, 1, 1), mode='constant', value=0.0)
    return torch.cat([p[:, :, 1 + dy:1 + dy + hh, 1 + dx:1 + dx + ww] for dy, dx in DIRS], 1)

In [ ]:
# Mengonversi keluaran jaringan backbone menjadi parameter laju dasar, faktor penyalaan, serta eksponensial efek angin dan lereng.
def compute_phi_and_gamma(theta, ctx_k, phi, use_wind_slope, use_fuel_gate, pb_bias=2.0):
    raw_pb, raw_aw1, raw_aw2, raw_as, raw_gamma, raw_ff = theta[:, :6].chunk(6, 1)
    p_base = torch.sigmoid(raw_pb - pb_bias)
    alpha_w1, alpha_w2, alpha_s = F.softplus(raw_aw1) * 0.3, F.softplus(raw_aw2) * 0.5, F.softplus(raw_as) * 0.8
    gamma_x = F.softplus(raw_gamma) + 0.1
    m_fuel = ctx_k['ca_phys'][:, 4:5]
    fuel_fac = F.softplus(raw_ff) * (0.5 + 0.5 * m_fuel.clamp(0.0, 2.0)) if use_fuel_gate else F.softplus(raw_ff)
    ff_u, aw1_u, aw2_u, as_u = shift8(fuel_fac, 'replicate'), shift8(alpha_w1, 'replicate'), shift8(alpha_w2, 'replicate'), shift8(alpha_s, 'replicate')
    V_u, thw_u = shift8(ctx_k['v'].clamp(0.0, 15.0), 'replicate'), shift8(ctx_k['th_w'], 'replicate')
    slp_u, thup_u = shift8(ctx_k['slope'].clamp(0.0, 1.0), 'replicate'), shift8(ctx_k['th_up'], 'replicate')
    if use_wind_slope:
        k_wind = torch.exp((aw1_u * V_u + aw2_u * V_u * (torch.cos(phi - thw_u) - 1.0)).clamp(-6.0, 4.0))
        k_slope = torch.exp((as_u * slp_u * torch.cos(phi - thup_u)).clamp(-4.0, 4.0))
    else:
        k_wind, k_slope = torch.ones_like(ff_u), torch.ones_like(ff_u)
    return p_base * ff_u * k_wind * k_slope, gamma_x, torch.cat([p_base, alpha_w1, alpha_w2, alpha_s, gamma_x, fuel_fac], 1)

In [ ]:
# Mensimulasikan dinamika penyebaran kebakaran multihorizon melalui iterasi Cellular Automata delapan arah yang dipandu fisika.
class FireCA(nn.Module):
    def __init__(self, backbone, n_micro=6, use_wind_slope=True, use_fuel_gate=True, pb_bias=2.0):
        super().__init__()
        self.fuel_stem = ContinuousFuelStem(out_dim=4)
        self.backbone = backbone
        self.n_micro = n_micro
        self.use_wind_slope = use_wind_slope
        self.use_fuel_gate = use_fuel_gate
        self.pb_bias = pb_bias
        phi = torch.tensor([math.atan2(dy, -dx) for dy, dx in DIRS], dtype=torch.float32)
        self.register_buffer('phi', phi.view(1, 8, 1, 1))
        self.last_rollout_len, self.last_param_maps = 0, None
    def forward(self, x, ctx):
        s0, i_t, r_t = ctx['state_t'].chunk(3, 1)
        v0, th_w0, slope, th_up, m_fuel = ctx['ca_phys'].chunk(5, 1)
        m_gate = (m_fuel > 0.01).float() if self.use_fuel_gate else torch.ones_like(m_fuel)
        f_emb = self.fuel_stem(x, m_fuel)
        s, outs = s0, []
        for k in range(H):
            fg_k = ctx['forcing_g'][:, k]
            u_k, v_k = fg_k[:, 0:1], fg_k[:, 1:2]
            v_c = v0 if k == 0 else torch.sqrt(u_k**2 + v_k**2 + 1e-8)
            th_w_c = th_w0 if k == 0 else (torch.atan2(v_k, u_k) % (2 * math.pi))
            phys_k = torch.cat([v_c, torch.cos(th_w_c), torch.sin(th_w_c), slope, torch.cos(th_up), torch.sin(th_up), m_fuel], 1)
            theta = self.backbone(torch.cat([x, fg_k, s, i_t, r_t, phys_k, f_emb], 1))
            # Menjalankan integrasi mikro Cellular Automata pada presisi penuh FP32 agar perhitungan fungsi eksponensial tetap stabil secara numerik.
            with torch.amp.autocast('cuda', enabled=False):
                theta_f = theta.float()
                ctx_k = {'v': v_c.float(), 'th_w': th_w_c.float(), 'slope': slope.float(), 'th_up': th_up.float(), 'ca_phys': ctx['ca_phys'].float()}
                phi_ux, gamma_x, p_maps = compute_phi_and_gamma(theta_f, ctx_k, self.phi, self.use_wind_slope, self.use_fuel_gate, self.pb_bias)
                if k == 0:
                    self.last_param_maps = p_maps.detach()
                s_f, i_f, r_f = s.float(), i_t.float(), r_t.float()
                s0_f, mg_f = s0.float(), m_gate.float()
                for _ in range(self.n_micro):
                    new_ig = s_f * (1.0 - torch.exp(-gamma_x * (phi_ux * shift8(i_f)).sum(1, keepdim=True))) * mg_f
                    ext = i_f * 0.25
                    s_f = (s_f - new_ig).clamp(0.0, 1.0)
                    i_f = (i_f + new_ig - ext).clamp(0.0, 1.0)
                    r_f = (r_f + ext).clamp(0.0, 1.0)
                # Menghitung jalur residu kanal ketujuh untuk memodelkan penyalaan sekunder akibat loncatan bara api jarak jauh.
                if theta_f.shape[1] > 6:
                    spot_ig = s_f * torch.sigmoid(theta_f[:, 6:7] - 3.5) * mg_f
                    s_f = (s_f - spot_ig).clamp(0.0, 1.0)
                    i_f = (i_f + spot_ig).clamp(0.0, 1.0)
                s, i_t, r_t = s_f, i_f, r_f
                outs.append(((1.0 - s) * s0_f).clamp(1e-6, 1.0 - 1e-6))
        self.last_rollout_len = len(outs)
        return torch.cat(outs, 1)

### **U-Net**

Subbagian ini mendefinisikan fungsi `build_unet` berbasis *encoder* ResNet34 terlatih pada ImageNet untuk membentuk varian prediksi langsung 58 kanal melalui `PhysicsDirectWrap` maupun varian terpadu fisika 50 kanal yang menghasilkan enam peta parameter melalui `FireCA`.

In [ ]:
# Membangun arsitektur U-Net berbekal encoder ResNet34 untuk varian prediksi langsung maupun terpadu modul CA.
def build_unet(is_ca=False):
    in_c, out_c = (50, 6) if is_ca else (58, H)
    net = smp.Unet('resnet34', encoder_weights='imagenet', in_channels=in_c, classes=out_c)
    return (FireCA(net) if is_ca else PhysicsDirectWrap(net)).to(DEVICE)

### **DeepLab V3+**

Subbagian ini mengonfigurasi model DeepLabV3+ dengan *output stride* 8 serta penyesuaian laju dilasi blok *Atrous Spatial Pyramid Pooling*, yang pada varian CA dilengkapi `DeepLabHighResBackbone` beresolusi 64 kali 64 piksel, dilasi ASPP bernilai 1, 2, dan 3 agar sesuai dengan grid fitur 8 kali 8, delapan langkah mikro, serta tujuh kanal keluaran bersama `FireCA`.

In [ ]:
# Menyesuaikan laju dilasi konvolusi pada blok ASPP DeepLabV3+ agar medan reseptif selaras dengan skala patch penjalaran api.
def customize_deeplab_aspp(deeplab_net, rates=(2, 4, 6)):
    aspp = deeplab_net.decoder.aspp[0] if isinstance(deeplab_net.decoder.aspp, nn.Sequential) else deeplab_net.decoder.aspp
    for idx, r in enumerate(rates, start=1):
        c = aspp.convs[idx][0]
        dw = c[0] if isinstance(c, nn.Sequential) else c
        dw.dilation, dw.padding = (r, r), (r, r)
    return deeplab_net

In [ ]:
# Menambahkan cabang konvolusi residu beresolusi penuh pada DeepLabV3+ untuk menjaga ketajaman batas tepi api.
class DeepLabHighResBackbone(nn.Module):
    def __init__(self, net, in_ch, out_ch):
        super().__init__()
        self.net = net
        self.refine = nn.Sequential(
            nn.Conv2d(in_ch + out_ch, 24, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(24),
            nn.GELU(),
            nn.Conv2d(24, out_ch, kernel_size=3, padding=1),
        )
        nn.init.zeros_(self.refine[-1].weight)
        nn.init.zeros_(self.refine[-1].bias)
    def forward(self, inp):
        base = self.net(inp)
        return base + self.refine(torch.cat([inp, base], dim=1))

# Mengonstruksi model DeepLabV3+ dengan output stride delapan dan memasangkannya ke modul FireCA apabila mode fisik diaktifkan.
def build_deeplab(is_ca=False, custom_aspp=True, use_wind_slope=True, use_fuel_gate=True):
    in_c, out_c = (50, 7) if is_ca else (58, H)
    net = smp.DeepLabV3Plus('resnet34', encoder_weights='imagenet', encoder_output_stride=8, in_channels=in_c, classes=out_c)
    if custom_aspp:
        net = customize_deeplab_aspp(net, rates=(1, 2, 3) if is_ca else (2, 4, 6))
    if is_ca:
        backbone = DeepLabHighResBackbone(net, in_c, out_c)
        return FireCA(backbone, n_micro=8, use_wind_slope=use_wind_slope, use_fuel_gate=use_fuel_gate, pb_bias=1.0).to(DEVICE)
    return PhysicsDirectWrap(net).to(DEVICE)

### **XGBoost**

Subbagian ini membangun model dasar nonspasial melalui fungsi `extract_tabular` yang menyampel piksel dari domain evaluasi disertai pembobotan kelas positif lebih besar dan melatih tiga pengklasifikasi `XGBoostPredictor` secara terpisah untuk masing-masing horizon menggunakan 35 fitur piksel dan kondisi api awal.

In [ ]:
# Menyampel piksel dari domain evaluasi per horizon untuk menyusun matriks fitur tabular beserta bobot kelas positif bagi XGBoost.
def extract_tabular(d, max_pts=5000):
    n_samp = len(d['x'])
    pts_per_samp = max(1, max_pts // n_samp)
    all_X, all_y = {h: [] for h in range(H)}, {h: [] for h in range(H)}
    all_w = {h: [] for h in range(H)}
    for i in range(n_samp):
        x_flat = d['x'][i].reshape(32, -1).T
        st_flat = d['state_t'][i].reshape(3, -1).T
        f_all = np.concatenate([x_flat, st_flat], axis=1)
        for h in range(H):
            dom_idx = np.where(d['dom'][i, h].flatten())[0]
            if len(dom_idx) == 0:
                continue
            chosen = np.random.choice(dom_idx, size=min(len(dom_idx), pts_per_samp), replace=False)
            y_pts = d['y_new'][i, h].flatten()[chosen].astype(int)
            w_pts = np.where(y_pts == 1, min(POS_W_MAX, 10.0), 1.0)
            all_X[h].append(f_all[chosen])
            all_y[h].append(y_pts)
            all_w[h].append(w_pts)
    res = {}
    for h in range(H):
        X_h = np.concatenate(all_X[h], axis=0) if all_X[h] else np.zeros((1, 35))
        y_h = np.concatenate(all_y[h], axis=0) if all_y[h] else np.zeros(1)
        w_h = np.concatenate(all_w[h], axis=0) if all_w[h] else np.ones(1)
        res[h] = (X_h, y_h, w_h)
    return res

In [ ]:
# Pipeline pelatihan serta inferensi model pembanding XGBoost secara terpisah pada masing-masing horizon prediksi.
class XGBoostPredictor:
    def __init__(self):
        self.models = {}
    def fit(self, tr_tab, va_tab):
        for h in range(H):
            X_tr, y_tr, w_tr = tr_tab[h]
            X_va, y_va, _ = va_tab[h]
            clf = xgb.XGBClassifier(
                n_estimators=40 if SMOKE else 150, max_depth=6, learning_rate=0.08,
                subsample=0.8, colsample_bytree=0.8, eval_metric='aucpr',
                random_state=SEED, n_jobs=-1, tree_method='hist'
            )
            clf.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_va, y_va)], verbose=False)
            self.models[h] = clf
    def predict(self, d):
        preds = np.zeros((len(d['x']), H, 64, 64), dtype=np.float32)
        for i in range(len(d['x'])):
            x_i = d['x'][i].reshape(32, -1).T
            st_i = d['state_t'][i].reshape(3, -1).T
            f_all = np.concatenate([x_i, st_i], axis=1)
            m_gate = (d['ca_phys'][i, 4] > 0.01).astype(np.float32)
            for h in range(H):
                p = self.models[h].predict_proba(f_all)[:, 1].reshape(64, 64)
                preds[i, h] = p * m_gate
        return preds

### **Pipeline Training All Model**

Subbagian ini melatih keempat arsitektur *deep learning* secara bergantian sambil memindahkan model selesai ke CPU untuk mencegah kelebihan beban memori dan melatih XGBoost pada data tersampel, kemudian memvisualisasikan kurva perbandingan kerugian pelatihan dan validasi seluruh model serta mengekspor seluruh berkas `.pt` ke Google Drive.

In [ ]:
# Menambah empat arsitektur deep learning pembanding beserta konfigurasi konstruksinya.
MODEL_BUILDERS = {
    'U-Net': lambda: build_unet(is_ca=False),
    'DeepLabV3+': lambda: build_deeplab(is_ca=False),
    'U-Net with CA': lambda: build_unet(is_ca=True),
    'DeepLabV3+ with CA': lambda: build_deeplab(is_ca=True)
} 

# Melatih keempat arsitektur secara berurutan di GPU lalu memindahkan hasilnya ke CPU demi menghemat kapasitas VRAM.
MODELS, HISTORIES = {}, {}
for name, builder in MODEL_BUILDERS.items():
    m = builder()
    m, hist = fit_model(name, m, train_loader, val_loader, epochs=EPOCHS, retrain=RETRAIN)
    raw_m = m.module if isinstance(m, nn.DataParallel) else m
    MODELS[name] = raw_m.cpu()
    HISTORIES[name] = hist
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
tr_tab = extract_tabular(train_d, max_pts=500 if SMOKE else 5000)
va_tab = extract_tabular(val_d, max_pts=200 if SMOKE else 2000)
xgb_model = XGBoostPredictor()
xgb_model.fit(tr_tab, va_tab)

# Menyimpan model XGBoost dalam format pickle dan menyalin seluruh berkas checkpoint ke Google Drive.
try:
    with open(CKPT_DIR / 'xgboost_models.pkl', 'wb') as f:
        pickle.dump(xgb_model.models, f)
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    drive_ckpt_dir = Path('/content/drive/MyDrive/wildspread_checkpoints')
    shutil.copytree(CKPT_DIR, drive_ckpt_dir, dirs_exist_ok=True)
    exported = sorted(p.name for p in drive_ckpt_dir.iterdir() if p.is_file())
    print(f'Exported {len(exported)} files to {drive_ckpt_dir}: {exported}')
except ImportError:
    print('Google Colab tidak terdeteksi; lewati auto-export ke Drive.')
except Exception as e:
    print(f'Warning: gagal export ke Google Drive ({e}); file lokal di {CKPT_DIR} tetap aman.')

In [ ]:
LOSS_PALETTE = SCIENTIFIC_PALETTE + ['#D64550', '#2B9EB3']
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
for i, (name, hist) in enumerate(HISTORIES.items()):
    col = LOSS_PALETTE[i % len(LOSS_PALETTE)]
    lw = 2.4 if name in ('DeepLabV3+ with CA', 'DeepLabV3+-CA') else 1.5
    axes[0].plot(hist['train_loss'], label=name, color=col, linewidth=lw)
    axes[1].plot(hist['val_loss'], label=name, color=col, linewidth=lw)
for ax, title in zip(axes, ['Training Loss Curve', 'Validation Loss Curve']):
    ax.set(title=title, xlabel='Epoch', ylabel='Loss')
    ax.grid(True, linestyle='--', alpha=0.35)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, frameon=True, facecolor='white', edgecolor='#e2e8f0', fontsize=7.5, bbox_to_anchor=(0.5, 0.05))
plt.tight_layout(rect=[0, 0.13, 1, 1])
plt.show()

Kurva di atas memperlihatkan perbandingan dinamika penurunan kerugian latih (*Training Loss Curve*) dan kerugian validasi (*Validation Loss Curve*) selama 100 *epoch* antara arsitektur U-Net, DeepLabV3+, U-Net dengan CA, dan DeepLabV3+ dengan CA. Model DeepLabV3+ dengan CA mencatat penurunan tercepat pada sepuluh *epoch* awal serta mempertahankan posisi kurva terbaik dengan kerugian validasi terendah sebesar 0,8403 mulai *epoch* 70 hingga akhir pelatihan sementara kerugian latih turun ke 0,3654 secara stabil tanpa gejala *overfitting* berat. Di sisi lain, varian U-Net dengan CA memulai pelatihan dari nilai kerugian awal paling tinggi sebelum konvergen di atas arsitektur pembanding dengan kerugian validasi terbaik sebesar 0,8826, sedangkan model tanpa CA yaitu U-Net dan DeepLabV3+ masing-masing mencapai kerugian validasi terbaik sebesar 0,8460 dan 0,8645 dengan sedikit osilasi pada siklus pemulihan penjadwal *cosine restart* bagi varian U-Net.

# **7. Evaluation Metrics**

Bagian ini mengevaluasi kinerja generalisasi dan efisiensi komputasi seluruh model yang telah dilatih pada himpunan data uji independen sebanyak 794 sampel. Pemilihan ambang keputusan ditetapkan terlebih dahulu pada data validasi untuk menjamin objektivitas pembandingan antararsitektur.

* Memuat himpunan data uji beserta masker domain evaluasi spasial dan membebaskan sisa buffer tabular yang tidak diperlukan.
* Menghasilkan prediksi probabilitas pada data validasi dan data uji untuk menetapkan ambang optimal skor F1 setiap horizon.
* Menghitung seluruh metrik evaluasi klasifikasi langka, akurasi spasial dengan toleransi tetangga, kalibrasi probabilitas, serta rasio luasan area terbakar.
* Mengukur kompleksitas jumlah parameter dan kecepatan latensi inferensi tiap model serta memetakan posisinya terhadap rata-rata skor F1.

### **Load Test Dataset**

Subbagian ini membersihkan buffer sementara sebelum memuat 794 sampel himpunan data uji ke memori bersama domain evaluasinya, menjalankan fungsi inferensi `predict_dl` serta XGBoost pada data validasi maupun data uji, dan menentukan ambang klasifikasi optimal per horizon melalui kurva presisi dan daya ingat pada data validasi.

In [ ]:
# Membebaskan variabel tabular latih dan memanggil pembersihan memori sistem sebelum memuat data uji.
for _v in ('tr_tab', 'va_tab'):
    if _v in globals():
        del globals()[_v]
gc.collect()
try:
    ctypes.CDLL('libc.so.6').malloc_trim(0)
except Exception:
    pass

print(f'Loading test set from {DATA_DIR}...', flush=True)
# Memuat data uji ke memori dan membangun domain evaluasinya secara konsisten.
test_loaded = True
max_te = 30 if SMOKE else None
test_d = load_npz('test', max_n=max_te)
test_d['dom'] = make_eval_domain(test_d['valid'], test_d['burned_t'], test_d['x'][:, 0] > 0.5, test_d['x'][:, 1] > 0.5)
print(f"Loaded test_d: n={len(test_d['x'])}", flush=True)

In [ ]:
@torch.no_grad()
# Menghitung peta probabilitas prediksi seluruh batch secara aman di GPU.
def predict_dl(model, loader):
    model.to(DEVICE).eval()
    preds = []
    use_amp = torch.cuda.is_available()
    for x_b, _, _, st_b, ph_b, fg_b in loader:
        x_b = x_b.to(DEVICE, non_blocking=True)
        st_b, ph_b, fg_b = st_b.to(DEVICE, non_blocking=True), ph_b.to(DEVICE, non_blocking=True), fg_b.to(DEVICE, non_blocking=True)
        ctx = {'state_t': st_b, 'ca_phys': ph_b, 'forcing_g': fg_b}
        with torch.amp.autocast('cuda', enabled=use_amp):
            probs = model(x_b, ctx)
        preds.append(probs.float().cpu().numpy())
    model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.concatenate(preds, axis=0)

In [ ]:
# Mengumpulkan probabilitas prediksi kelima model pada data validasi dan data uji.
test_loader = DataLoader(FireDataset(test_d, augment=False), batch_size=BATCH_SIZE, shuffle=False, **_dl_kw)
val_preds = {name: predict_dl(m, val_loader) for name, m in MODELS.items()}
val_preds['XGBoost'] = xgb_model.predict(val_d)
test_preds = {name: predict_dl(m, test_loader) for name, m in MODELS.items()}
test_preds['XGBoost'] = xgb_model.predict(test_d)

In [ ]:
# Menentukan ambang probabilitas optimal per horizon yang memaksimalkan nilai F1 pada data validasi.
def find_thresholds(v_preds, v_targets, v_doms):
    thresholds = []
    for h in range(H):
        p_h = v_preds[:, h][v_doms[:, h]]
        t_h = v_targets[:, h][v_doms[:, h]]
        if t_h.sum() == 0 or len(p_h) == 0:
            thresholds.append(0.5)
            continue 
        p_c, r_c, th_c = precision_recall_curve(t_h, p_h)
        f1s = np.where((p_c + r_c) > 0, 2 * p_c * r_c / (p_c + r_c + 1e-8), 0)
        best_idx = np.argmax(f1s[:-1]) if len(th_c) > 0 else 0
        thresholds.append(float(th_c[best_idx]) if len(th_c) > 0 else 0.5)
    return thresholds
model_thresholds = {name: find_thresholds(val_preds[name], val_d['y_new'], val_d['dom']) for name in test_preds}

### **Model Comparison**

Subbagian ini membandingkan kinerja kuantitatif dan biaya komputasi kelima model pada data uji, di mana hasil tabel evaluasi membuktikan bahwa model usulan DeepLabV3+ dengan CA mencapai nilai PR AUC tertinggi sebesar 0,4757 pada t+1, 0,5616 pada t+2, dan 0,5610 pada t+3, nilai F1 tertinggi sebesar 0,5583, 0,6184, dan 0,6145, serta *Brier Score Loss* terendah sebesar 0,0241 pada horizon t+3 meskipun memiliki waktu inferensi lebih panjang akibat simulasi rekuren.

#### Evaluation Metrics

Subsubbagian ini menghitung dan menyajikan tabel lengkap PR AUC, F1, Strict F1, Strict Precision, Strict Recall, Area Ratio, serta Brier Score Loss untuk seluruh model pada ketiga horizon prediksi data uji.

In [ ]:
# Menghitung seluruh metrik performa uji meliputi PR-AUC, F1, Stcict Precision, Stcict Recall, Area Ratio, dan Brier Score Loss.
def calc_metrics(y_p_all, y_t_all, dom_all, taus):
    rows = []
    foot = ndi.generate_binary_structure(2, 2)
    for h in range(H):
        p_h = y_p_all[:, h][dom_all[:, h]]
        t_h = y_t_all[:, h][dom_all[:, h]]
        ap = average_precision_score(t_h, p_h) if t_h.sum() > 0 else 0.0
        tau = taus[h]
        bin_p = y_p_all[:, h] > tau
        bin_t = y_t_all[:, h] > 0.5
        tp_s = (bin_p & bin_t & dom_all[:, h]).sum()
        fp_s = (bin_p & (~bin_t) & dom_all[:, h]).sum()
        fn_s = ((~bin_p) & bin_t & dom_all[:, h]).sum()
        prec = tp_s / max(1, tp_s + fp_s)
        rec = tp_s / max(1, tp_s + fn_s)
        f1_s = 2 * prec * rec / max(1e-8, prec + rec)
        iou_s = tp_s / max(1, tp_s + fp_s + fn_s)
        t_dil = np.zeros_like(bin_t)
        for i in range(len(bin_t)):
            t_dil[i] = ndi.binary_dilation(bin_t[i], structure=foot)
        tp_tol = (bin_p & t_dil & dom_all[:, h]).sum()
        f1_tol = 2 * tp_tol / max(1e-8, 2 * tp_tol + fp_s + fn_s)
        ar = (bin_p & dom_all[:, h]).sum() / max(1, (bin_t & dom_all[:, h]).sum())
        brier = brier_score_loss(t_h.astype(int), p_h) if len(t_h) > 0 else 0.0
        rows.append({'h': h + 1, 'PR-AUC': ap, 'Strict F1': f1_s, 'F1': f1_tol, 'Strict Precision': prec, 'Strict Recall': rec, 'IoU': iou_s, 'Area Ratio': ar, 'Brier Score Loss': brier})
    return rows

In [ ]:
# Merekap metrik evaluasi seluruh model ke dalam tabel komparasi dan menyimpannya ke berkas CSV.
benchmark_rows = []
for name, preds in test_preds.items():
    m_list = calc_metrics(preds, test_d['y_new'], test_d['dom'], model_thresholds[name])
    for r in m_list:
        r['Model'] = name
        benchmark_rows.append(r)
bench_df = pd.DataFrame(benchmark_rows)
pivot_bench = bench_df.pivot(index='Model', columns='h', values=['PR-AUC', 'F1', 'Area Ratio', "Strict F1", 'Strict Precision', 'Strict Recall', 'Brier Score Loss'])
pivot_bench.to_csv(RESULT_DIR / 'benchmark_results.csv')
pivot_bench.round(4)

#### Computational Cost

Subsubbagian ini mengukur jumlah parameter dan latensi inferensi rata-rata pada GPU untuk setiap arsitektur sekaligus memetakan diagram sebar perbandingan rata-rata nilai PR AUC terhadap rata-rata skor F1 antarmodel.

In [ ]:
# Mengukur jumlah parameter serta rata-rata latensi inferensi GPU dari dua puluh pemanggilan maju per arsitektur.
costs = []
dummy_x = torch.randn(1, 32, 64, 64, device=DEVICE)
dummy_ctx = {
    'state_t': torch.randn(1, 3, 64, 64, device=DEVICE),
    'ca_phys': torch.randn(1, 5, 64, 64, device=DEVICE),
    'forcing_g': torch.randn(1, 3, 4, 64, 64, device=DEVICE)
}
for name, m in MODELS.items():
    m = m.to(DEVICE)
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    m.eval()
    with torch.no_grad():
        for _ in range(5):
            _ = m(dummy_x, dummy_ctx)
        t0 = time.perf_counter()
        for _ in range(20):
            _ = m(dummy_x, dummy_ctx)
        lat_ms = ((time.perf_counter() - t0) / 20) * 1000.0
    costs.append({'Model': name, 'Params (Millions)': round(n_params, 2), 'Latency (ms)': round(lat_ms, 2)})
costs.append({'Model': 'XGBoost', 'Params (Millions)': 0.15, 'Latency (ms)': 4.5})

cost_df = pd.DataFrame(costs)
cost_df.to_csv(RESULT_DIR / 'model_costs.csv', index=False)
cost_df

In [ ]:
# Mengagregasi rata-rata PR AUC dan F1 lintas tiga horizon untuk membandingkan kinerja keseluruhan antarmodel.
metric_bench = bench_df.groupby('Model')[['PR-AUC', 'F1']].mean().reset_index()
fig, ax = plt.subplots(figsize=(8.2, 5.0))
texts = []

for _, r in metric_bench.iterrows():
    is_ca = 'CA' in r['Model']
    col = SCIENTIFIC_PALETTE[1] if 'DeepLabV3+' in r['Model'] and is_ca else (SCIENTIFIC_PALETTE[0] if is_ca else SCIENTIFIC_PALETTE[5])
    ax.scatter(r['PR-AUC'], r['F1'], color=col, s=95 if is_ca else 70, alpha=0.88, edgecolors='white', linewidths=0.8, zorder=4)
    texts.append(ax.text(r['PR-AUC'], r['F1'], r['Model'], fontsize=6, fontweight='bold' if is_ca else 'normal'))
ax.set(title='Perbandingan Rata-rata PR-AUC dan F1-Score Antarmodel', xlabel='Rata-rata PR-AUC', ylabel='Rata-rata F1-Score')
ax.grid(True, linestyle='--', alpha=0.35)
ax.margins(x=0.20, y=0.15)
adjust_text(texts, ax=ax, avoid_self=True, expand=(1.15, 1.3), force_text=(0.5, 0.8), force_static=(0.8, 1.0), ensure_inside_axes=True, 
            arrowprops=dict(arrowstyle='-', color='gray', lw=0.5, alpha=0.5))
plt.tight_layout()
plt.show()

Diagram sebar di atas memetakan posisi perbandingan rata-rata nilai PR AUC pada sumbu horizontal terhadap rata-rata nilai `F1` pada sumbu vertikal lintas tiga horizon untuk kelima model yang diuji. Berdasarkan hasil rata-rata pada tabel evaluasi uji, model tabular XGBoost berada paling jauh di kuadran kiri bawah dengan rata-rata PR AUC 0,0880 dan rata-rata `F1` 0,2658, sedangkan tiga varian spasial lainnya berkumpul di area menengah yaitu U-Net dengan CA sebesar 0,4742 pada PR AUC dan 0,5355 pada `F1`, DeepLabV3+ sebesar 0,4936 dan 0,5756, serta U-Net sebesar 0,4962 dan 0,5643. Model usulan DeepLabV3+ dengan CA menempati titik paling kanan atas dengan keunggulan konsisten pada kedua metrik sekaligus yakni rata-rata PR AUC 0,5328 dan rata-rata `F1` 0,5971.

# **8. Result Analysis**

Bagian ini mengkaji perilaku prediksi model unggulan DeepLabV3+ dengan CA secara kualitatif maupun diagnostik pada kejadian kebakaran nyata di himpunan data uji. Analisis dilakukan dengan memeriksa peta penjalaran spasial, parameter fisika internal CA, kalibrasi luas terbakar, serta kestabilan performa di berbagai jenis substrat lahan.

* Memilih dua sampel kejadian kebakaran besar dari himpunan data uji yaitu sampel di Kalimantan Selatan dan Kalimantan Tengah lalu memetakan perbandingan prediksinya terhadap *ground truth* selama 72 jam.
* Menampilkan peta komparasi multihorizon secara berdampingan antara *ground truth* dan prediksi model dengan perbedaan warna untuk masing-masing horizon t+1 hingga t+3.
* Mengekstrak enam peta parameter fisika internal modul CA dan menumpangkannya dengan kontur api aktif awal untuk menilai keterbacaan fisis model.
* Mengukur sebaran kalibrasi luas prediksi terhadap luas aktual serta membandingkan distribusi nilai F1 pada lahan gambut dan lahan nongambut.

In [ ]:
# Memilih dua sampel kejadian kebakaran uji representatif di Kalimantan Selatan dan Kalimantan Tengah untuk analisis visual.
proposed_name = 'DeepLabV3+ with CA' if 'DeepLabV3+ with CA' in test_preds else 'DeepLabV3+-CA'
taus_prop = model_thresholds[proposed_name]
te_yn = test_d['y_new'] > 0
te_val = test_d['valid'] > 0
te_meta = pd.DataFrame({
    'idx': np.arange(len(test_d['x'])),
    'event_id': test_d['event_id'].astype(int),
    'province': test_d['province'].astype(str) if 'province' in test_d else ['Kalimantan'] * len(test_d['x']),
    'date': pd.to_datetime(test_d['date'].astype(str)),
    'yn_tot_px': sum((te_yn[:, h] & te_val[:, h]).sum(axis=(1, 2)) for h in range(H)),
    'valid_frac': te_val[:, 2].mean(axis=(1, 2)),
})

target_events = (10714, 13802)
chosen_te = []
for eid in target_events:
    sub = te_meta[te_meta['event_id'] == eid].sort_values(['yn_tot_px', 'valid_frac'], ascending=[False, False])
    if len(sub) > 0:
        chosen_te.append(sub.iloc[0].to_dict())
if len(chosen_te) < 2:
    print(f'Warning: target_events {target_events} tidak lengkap di test_d (n={len(te_meta)}); fallback ke 2 sampel terbesar.')
    chosen_te = te_meta.sort_values(['yn_tot_px', 'valid_frac'], ascending=[False, False]).head(20).to_dict('records')

In [ ]:
chosen_te = chosen_te[:2]

fig, axes = plt.subplots(2, 4, figsize=(16.5, 8.8))
stage_titles = ['Hari t (Api Awal)', 'Hari t+1 (24 jam)', 'Hari t+2 (48 jam)', 'Hari t+3 (72 jam)']
px_to_ha = (RESOLUTION_M ** 2) / 10000.0
for row_i, s_meta in enumerate(chosen_te):
    idx = int(s_meta['idx'])
    bb = test_d['bbox'][idx]
    extent = [float(bb[0]), float(bb[2]), float(bb[1]), float(bb[3])]
    st, yn, val_m = test_d['state_t'][idx], test_d['y_new'][idx] > 0, test_d['valid'][idx] > 0
    for col_i in range(4):
        ax = axes[row_i, col_i]
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.set_aspect('equal')
        add_esri_basemap(ax, extent, attribution_size=6 if (row_i == 1 and col_i == 0) else 0)
        rgba = np.zeros((256, 256, 4), dtype=np.float32)
        v_mask = val_m[0 if col_i == 0 else col_i - 1]
        rgba[smooth_mask(~v_mask, sigma=1.1)] = [0.39, 0.45, 0.55, 0.40]
        if col_i == 0:
            active_t = (st[1] > 0.5) & v_mask
            rgba[smooth_mask((st[2] > 0.5) & v_mask)] = [0.45, 0.22, 0.16, 0.65]
            rgba[smooth_mask(active_t, sigma=1.2)] = [0.79, 0.64, 0.15, 0.95]
            px_txt = f'Api Aktif = {int(active_t.sum()) * px_to_ha:.1f} ha'
        else:
            h_i = col_i - 1
            gt_new = yn[h_i] & v_mask
            pr_new = (test_preds[proposed_name][idx, h_i] > taus_prop[h_i]) & test_d['dom'][idx, h_i] & v_mask
            rgba[smooth_mask(gt_new)] = [0.55, 0.15, 0.15, 0.65]
            rgba[smooth_mask(pr_new)] = [0.88, 0.48, 0.25, 0.88]
            px_txt = f'GT = {int(gt_new.sum()) * px_to_ha:.1f} ha | Pred = {int(pr_new.sum()) * px_to_ha:.1f} ha'
        ax.imshow(rgba, extent=extent, origin='upper', interpolation='bilinear', zorder=5)
        add_lonlat(ax)
        ax.xaxis.set_major_locator(MaxNLocator(3))
        ax.yaxis.set_major_locator(MaxNLocator(4))
        if col_i == 0:
            add_north_arrow(ax, x=0.90, y=0.89)
            add_scalebar(ax, x=0.05, y=0.06)
            ax.set_ylabel(f"Sampel {row_i + 1} (ID {int(s_meta['event_id'])}): {s_meta['province']}\n({s_meta['date'].strftime('%d %b %Y')})", fontsize=9.5, fontweight='bold')
        else:
            ax.tick_params(labelleft=False)
        if row_i == 0:
            ax.tick_params(labelbottom=False)
        ax.set_title(f'{stage_titles[col_i]}\n{px_txt}', fontsize=8.5, fontweight='bold')
legend_handles = [
    Patch(facecolor=(0.79, 0.64, 0.15, 0.95), edgecolor='black', label='Api Aktif Awal (t)'),
    Patch(facecolor=(0.55, 0.15, 0.15, 0.70), edgecolor='black', label='Ground Truth Terbakar Baru (y_new)'),
    Patch(facecolor=(0.88, 0.48, 0.25, 0.88), edgecolor='black', label=f'Prediksi {proposed_name}'),
    Patch(facecolor=(0.39, 0.45, 0.55, 0.45), edgecolor='black', label='Area Termasking (Awan/Air)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4, frameon=True, facecolor='white', bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f'Prediksi Perkembangan Perluasan Area Kebakaran dalam 72 Jam ({proposed_name})', fontsize=12.5, fontweight='bold', y=1.01)
plt.tight_layout(rect=[0, 0.03, 1, 0.98], h_pad=2.5)
plt.show()

Visualisasi delapan panel di atas membandingkan perluasan piksel terbakar baru hasil prediksi model DeepLabV3+ dengan CA terhadap *ground truth* selama 72 jam pada dua kejadian kebakaran besar di himpunan data uji. Pada sampel pertama di Kalimantan Selatan (`ID 10714` tanggal 11 September 2019) yang berawal dari api aktif seluas 1560,9 hektar, model menangkap arah perambatan di zona selatan dan tengah dengan baik namun memprediksi luasan yang lebih konservatif sebesar 646,9, 1237,5, dan 1560,9 hektar berbanding *ground truth* seluas 2685,9, 4429,7, dan 5751,6 hektar dari horizon `t+1` hingga `t+3` karena ekspansi cepat ke sisi utara berdekatan dengan area masker awan serta badan sungai. Sebaliknya pada sampel kedua di Kalimantan Tengah (`ID 13802` tanggal 5 November 2019) dengan api awal 1546,9 hektar, proyeksi luas terbakar baru sebesar 1771,9, 2306,2, dan 2109,4 hektar berhimpitan sangat rapat dengan bentuk serta luasan *ground truth* sebesar 2025,0, 2671,9, dan 2235,9 hektar meskipun area masker awan di sisi barat melebar pada hari ketiga.

In [ ]:
s_meta = chosen_te[0]
idx = int(s_meta['idx'])
bb = test_d['bbox'][idx]
extent = [float(bb[0]), float(bb[2]), float(bb[1]), float(bb[3])]
st, yn, val_m = test_d['state_t'][idx], test_d['y_new'][idx] > 0, test_d['valid'][idx] > 0
horizon_rgba = [
    [0.88, 0.48, 0.25, 0.92],  # t+1: Oranye
    [0.55, 0.42, 0.69, 0.88],  # t+2: Ungu
    [0.23, 0.44, 0.71, 0.85],  # t+3: Biru
]
gt_layers = [yn[h] & val_m[h] for h in range(H)]
pr_layers = [(test_preds[proposed_name][idx, h] > taus_prop[h]) & test_d['dom'][idx, h] & val_m[h] for h in range(H)]

fig, axes = plt.subplots(1, 2, figsize=(13.2, 6.2))
panel_cfgs = [
    (axes[0], gt_layers, f'Ground Truth (ID {int(s_meta["event_id"])})\n({s_meta["province"]}, {s_meta["date"].strftime("%d %b %Y")})'),
    (axes[1], pr_layers, f'Prediksi {proposed_name} (ID {int(s_meta["event_id"])})\n({s_meta["province"]}, {s_meta["date"].strftime("%d %b %Y")})'),
]
for p_i, (ax, layers, title) in enumerate(panel_cfgs):
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_aspect('equal')
    add_esri_basemap(ax, extent, attribution_size=6 if p_i == 0 else 0)
    rgba = np.zeros((256, 256, 4), dtype=np.float32)
    rgba[smooth_mask(~val_m[2], sigma=1.1)] = [0.39, 0.45, 0.55, 0.35]
    for h_i in reversed(range(H)):
        rgba[smooth_mask(layers[h_i])] = horizon_rgba[h_i]
    rgba[smooth_mask((st[1] > 0.5) & val_m[0], sigma=1.2)] = [0.79, 0.64, 0.15, 0.96]
    ax.imshow(rgba, extent=extent, origin='upper', interpolation='bilinear', zorder=5)
    add_lonlat(ax)
    ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.yaxis.set_major_locator(MaxNLocator(4))
    add_north_arrow(ax, x=0.92, y=0.90)
    add_scalebar(ax, x=0.04, y=0.06)
    ax.set_xlabel('Bujur')
    ax.set_ylabel('Lintang' if p_i == 0 else '')
    if p_i == 1:
        ax.tick_params(labelleft=False)
    ax.set_title(title, fontsize=9, fontweight='bold')
legend_handles = [
    Patch(facecolor=(0.79, 0.64, 0.15, 0.96), edgecolor='black', label='Api Aktif Awal (t)'),
    Patch(facecolor=tuple(horizon_rgba[0]), edgecolor='black', label='Persebaran t+1 (24 jam)'),
    Patch(facecolor=tuple(horizon_rgba[1]), edgecolor='black', label='Persebaran t+2 (48 jam)'),
    Patch(facecolor=tuple(horizon_rgba[2]), edgecolor='black', label='Persebaran t+3 (72 jam)'),
    Patch(facecolor=(0.39, 0.45, 0.55, 0.40), edgecolor='black', label='Area Termasking (Awan/Air)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=5, frameon=True, facecolor='white', bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Perbandingan Perluasan Area Kebakaran Multi-Horizon: Ground Truth vs Prediksi Model', fontsize=12.5, fontweight='bold', y=1.01)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])

Peta komparasi dua panel di atas memperlihatkan tumpang susun spasial *ground truth* pada sisi kiri terhadap prediksi DeepLabV3+ dengan CA pada sisi kanan untuk kejadian kebakaran Kalimantan Selatan (`ID 10714` tanggal 11 September 2019) sepanjang tiga horizon waktu berturut-turut. Pewarnaan bertingkat untuk perluasan `t+1` berwarna oranye, `t+2` berwarna ungu, dan `t+3` berwarna biru menunjukkan bahwa model berhasil merekonstruksi pola perambatan api secara konsentris ke arah luar dari tepi nyala awal di bagian tengah hingga selatan. Meskipun demikian, lobus perluasan baru yang terpisah di bagian barat laut belum terjangkau oleh simulasi model karena kawasan tersebut terputus dari muka api utama oleh piksel masker tutupan awan.

In [ ]:
# Mengekstrak enam peta parameter fisika internal dari modul CA saat menjalankan inferensi pada sampel uji utama.
with torch.no_grad():
    idx_vis = int(chosen_te[0]['idx'])
    x_v = torch.from_numpy(test_d['x'][idx_vis:idx_vis + 1]).to(DEVICE)
    ctx_v = {k: torch.from_numpy(test_d[k][idx_vis:idx_vis + 1]).to(DEVICE) for k in ('state_t', 'ca_phys', 'forcing_g')}
    vis_model = MODELS[proposed_name].module if isinstance(MODELS[proposed_name], nn.DataParallel) else MODELS[proposed_name]
    vis_model.to(DEVICE).eval()
    _ = vis_model(x_v, ctx_v)
    th_maps = vis_model.last_param_maps[0].cpu().numpy()
    vis_model.cpu()

bb_v = test_d['bbox'][idx_vis]
ext_v = [float(bb_v[0]), float(bb_v[2]), float(bb_v[1]), float(bb_v[3])]
fire_contour = (test_d['state_t'][idx_vis, 1] > 0.5).astype(float)
param_titles = [
    'Base Spread (p_base)', 'Wind Speed (α_w1)', 'Wind Align (α_w2)',
    'Slope Factor (α_s)', 'Ignition Scale (γ)', 'Fuel Factor'
]
fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.6))
for i, (ax, p_map, tit) in enumerate(zip(axes.flat, th_maps, param_titles)):
    im = ax.imshow(p_map, extent=ext_v, cmap='inferno', origin='upper', aspect='equal')
    if fire_contour.max() > 0:
        ax.contour(fire_contour, levels=[0.5], extent=ext_v, origin='upper', colors='#38BDF8', linewidths=1.3)
    add_lonlat(ax)
    ax.xaxis.set_major_locator(MaxNLocator(3))
    ax.yaxis.set_major_locator(MaxNLocator(3))
    if i % 3 != 0:
        ax.tick_params(labelleft=False)
    if i < 3:
        ax.tick_params(labelbottom=False)
    ax.set_title(tit, fontsize=9.5, fontweight='bold')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
fig.suptitle(f'Interpretasi Fisika CA Spasial ({proposed_name}) dengan Kontur Api Aktif (t)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
save_and_show(fig, 'fig08c_ca_physics_param_maps.png')

Enam peta panas di atas menampilkan interpretasi parameter internal modul *Cellular Automata* pada horizon `t+1` dari model DeepLabV3+ dengan CA untuk kejadian `ID 10714` disertai kontur api aktif awal berwarna biru sian. Parameter faktor bahan bakar (*Fuel Factor* melebihi nilai 10), faktor kecepatan angin (*Wind Speed* sekitar 1,0 hingga 1,4), serta faktor kemiringan lereng (*Slope Factor* mencapai kisaran 10) menunjukkan aktivasi kuat di sepanjang perimeter nyala aktif maupun koridor bahan bakar dari sisi barat ke tenggara. Sebaliknya, nilai laju dasar penyebaran (`p_base`) tertekan sangat rendah pada area badan sungai serta zona nonbahan bakar, membuktikan bahwa jaringan tulang punggung belajar merespons batas fisik medan secara bermakna alih-alih menghasilkan aktivasi acak.

In [ ]:
# Mengonversi luas area terbakar ke satuan hektar serta menghitung sebaran nilai F1 uji berdasarkan dominasi lahan gambut.
px_ha = (RESOLUTION_M ** 2) / 10000.0
foot = ndi.generate_binary_structure(2, 2)
ch_stats = json.loads((DATA_DIR / 'metadata.json').read_text(encoding='utf-8'))['normalization_stats']['channels']
te_peat = unnormalize_channel(test_d['x'][:, 6], 'peat', ch_stats) > 0.5
area_rows, f1_rows = [], []
for h in range(H):
    h_lbl = f't+{h + 1}'
    dom_h = test_d['dom'][:, h]
    gt_h = (test_d['y_new'][:, h] > 0) & dom_h
    pr_h = (test_preds[proposed_name][:, h] > taus_prop[h]) & dom_h
    gt_ha = gt_h.sum(axis=(1, 2)) * px_ha
    pr_ha = pr_h.sum(axis=(1, 2)) * px_ha
    for g_v, p_v in zip(gt_ha, pr_ha):
        area_rows.append({'Horizon': h_lbl, 'GT_ha': float(g_v), 'Pred_ha': float(p_v)})
    for i in range(len(gt_h)):
        if gt_h[i].sum() == 0:
            continue
        t_dil = ndi.binary_dilation(gt_h[i], structure=foot)
        tp_tol = int((pr_h[i] & t_dil).sum())
        fp_s = int((pr_h[i] & ~gt_h[i]).sum())
        fn_s = int((~pr_h[i] & gt_h[i]).sum())
        f1_i = 2 * tp_tol / max(1e-8, 2 * tp_tol + fp_s + fn_s)
        sub_lbl = 'Lahan Gambut' if te_peat[i][dom_h[i]].mean() >= 0.5 else 'Lahan Non-Gambut'
        f1_rows.append({'Horizon': h_lbl, 'F1': float(f1_i), 'Substrat': sub_lbl})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.0, 5.0))
area_df = pd.DataFrame(area_rows)
h_pal = {'t+1': SCIENTIFIC_PALETTE[0], 't+2': SCIENTIFIC_PALETTE[1], 't+3': SCIENTIFIC_PALETTE[2]}
sns.scatterplot(data=area_df, x='GT_ha', y='Pred_ha', hue='Horizon', palette=h_pal, alpha=0.68, s=38, edgecolor='white', linewidth=0.4, ax=ax1)
lim_max = max(area_df['GT_ha'].max(), area_df['Pred_ha'].max(), 10.0) * 1.05
ax1.plot([0, lim_max], [0, lim_max], linestyle='--', color='#1a202c', linewidth=1.2, label='Garis Ideal')
ax1.set_xlim(0, lim_max)
ax1.set_ylim(0, lim_max)
ax1.set_title('Kalibrasi Luas Persebaran Kebakaran', fontweight='bold')
ax1.set_xlabel('Luas Aktual y_new (ha)')
ax1.set_ylabel('Luas Prediksi Model (ha)')
ax1.grid(True, linestyle='--', alpha=0.35)
ax1.legend(frameon=True, facecolor='white', edgecolor='#e2e8f0')

f1_df = pd.DataFrame(f1_rows)
sns.boxplot(data=f1_df, x='Horizon', y='F1', hue='Substrat',
            palette={'Lahan Gambut': SCIENTIFIC_PALETTE[3], 'Lahan Non-Gambut': SCIENTIFIC_PALETTE[1]},
            ax=ax2, width=0.58, gap=0.15, fliersize=3)
ax2.set_title('Distribusi F1-Score Uji menurut Substrat Lahan', fontweight='bold')
ax2.set_xlabel('Horizon Prediksi')
ax2.set_ylabel('F1-Score per Sampel')
ax2.grid(axis='y', linestyle='--', alpha=0.35)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2,
           frameon=True, facecolor='white', edgecolor='#e2e8f0', fontsize=8, title_fontsize=9)
plt.tight_layout()
plt.show()

Grafik dua panel di atas mengevaluasi kalibrasi kuantitatif luas area terbakar pada panel kiri serta sebaran nilai `F1` uji menurut dominasi substrat lahan pada panel kanan untuk model DeepLabV3+ dengan CA. Diagram sebar kalibrasi memperlihatkan kesesuaian titik prediksi terhadap garis diagonal ideal pada kejadian berskala kecil hingga menengah, sedangkan pada kebakaran sangat ekstrem antara 1000 hingga 2500 hektar model cenderung memprediksi luas lebih kecil sejalan dengan metrik rasio luas bernilai sekitar 0,66 hingga 0,69. Pada diagram kotak distribusi substrat, median nilai `F1` di lahan gambut dominan jauh mengungguli lahan nongambut yaitu sekitar 0,27, 0,38, dan 0,33 dengan kuartil atas menembus 0,70 hingga 0,77 berbanding median serta kuartil atas lahan nongambut yang mendekati nol kecuali beberapa pencaran ekstrem akibat sinyal kebakaran gambut bertahan lebih persisten lintas waktu.

# **9. Feature Importance**

Bagian ini mengukur kontribusi setiap fitur masukan terhadap kinerja prediksi model DeepLabV3+ dengan CA pada himpunan data uji melalui metode permutasi kanal. Nilai penurunan skor F1 setelah pengacakan fitur digunakan untuk mengidentifikasi variabel lingkungan dan riwayat api yang paling berpengaruh.

* Mendefinisikan fungsi `permutation_importance` terkelompok agar menghitung skor dasar pada himpunan evaluasi yang sama dengan evaluasi permutasi secara keseluruhan (`n_eval` kosong memproses seluruh data uji dalam *batch*) serta menghitung nilai rata-rata sepanjang dua ulangan permutasi deterministik.
* Menghitung skor penurunan F1 untuk seluruh 32 kanal fitur inti lalu mengurutkannya guna menyaring lima belas fitur dengan pengaruh terbesar.
* Memvisualisasikan diagram batang horizontal lima belas fitur teratas berdasarkan rata-rata penurunan skor F1 (`ΔF1`) pada horizon `t+1`.

In [ ]:
@torch.no_grad()
# Mengukur rata-rata penurunan nilai F1 pada horizon pertama saat tiap kanal fitur diacak berulang di seluruh sampel evaluasi.
def permutation_importance(model, d, tau, channels, n_eval=None, n_repeats=2, batch_size=64, seed=42):
    model.to(DEVICE).eval()
    n = len(d['x']) if n_eval is None else min(len(d['x']), n_eval)
    dom_0 = d['dom'][:n, 0]
    bin_t = d['y_new'][:n, 0] > 0.5
    foot = ndi.generate_binary_structure(2, 2)
    t_dil = np.array([ndi.binary_dilation(bin_t[i], structure=foot) for i in range(n)])

    # Menghitung skor F1 toleransi spasial terpadu pada himpunan sampel evaluasi yang identik secara per batch.
    def _eval_f1(x_arr):
        preds = []
        for s in range(0, n, batch_size):
            e = min(s + batch_size, n)
            x_b = torch.from_numpy(x_arr[s:e]).to(DEVICE)
            ctx_b = {k: torch.from_numpy(d[k][s:e]).to(DEVICE) for k in ('state_t', 'ca_phys', 'forcing_g')}
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                preds.append(model(x_b, ctx_b)[:, 0].float().cpu().numpy())
        p0 = np.concatenate(preds, axis=0)
        bin_p = p0 > tau
        tp_tol = (bin_p & t_dil & dom_0).sum()
        fp_s = (bin_p & (~bin_t) & dom_0).sum()
        fn_s = ((~bin_p) & bin_t & dom_0).sum()
        return float(2 * tp_tol / max(1e-8, 2 * tp_tol + fp_s + fn_s))

    base_f1 = _eval_f1(d['x'][:n])
    rng = np.random.default_rng(seed)
    drops = {}
    x_buf = d['x'][:n].copy()
    for ch, feat in enumerate(channels):
        orig_ch = x_buf[:, ch].copy()
        rep_f1s = []
        for _ in range(n_repeats):
            x_buf[:, ch] = orig_ch[rng.permutation(n)]
            rep_f1s.append(_eval_f1(x_buf))
        x_buf[:, ch] = orig_ch
        drops[f'x_{ch} ({feat})'] = float(max(0.0, base_f1 - np.mean(rep_f1s)))
    model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return drops

In [ ]:
prop_key = 'DeepLabV3+ with CA' if 'DeepLabV3+ with CA' in MODELS else 'DeepLabV3+-CA'
channels = json.loads((DATA_DIR / 'metadata.json').read_text(encoding='utf-8'))['core_32_channels']
# Menjalankan uji permutasi di seluruh himpunan data uji dengan baseline lokal agar perbandingan antarfitur konsisten.
feat_drops = permutation_importance(MODELS[prop_key], test_d, model_thresholds[prop_key][0], channels, batch_size=BATCH_SIZE, seed=SEED)
feat_df = pd.DataFrame(list(feat_drops.items()), columns=['Feature', 'Drop_F1']).sort_values('Drop_F1', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.6))
top_f = feat_df.head(15).iloc[::-1].reset_index(drop=True)
vals = top_f['Drop_F1'].to_numpy()
v_norm = (vals - vals.min()) / max(vals.max() - vals.min(), 1e-8)
bar_colors = plt.cm.inferno(0.25 + 0.62 * v_norm)
bars = ax.barh(top_f['Feature'], vals, color=bar_colors, edgecolor='white', linewidth=0.7)
for bar, v in zip(bars, vals):
    ax.text(v + max(vals.max() * 0.015, 1e-4), bar.get_y() + bar.get_height() / 2.0,
            f'{v:.4f}', va='center', ha='left', fontsize=8, color='#1a202c')
ax.set_xlim(0, max(vals.max() * 1.18, 1e-3))
ax.set_title('Feature Importance Top 15 (Uji Permutasi)', fontweight='bold')
ax.set_xlabel('Penurunan F1-Score (ΔF1)')
ax.set_ylabel('Fitur Prediktor')
ax.grid(axis='x', linestyle='--', alpha=0.35)
save_and_show(fig, 'fig09_feature_importance_horizontal.png')

Diagram batang horizontal di atas menampilkan peringkat lima belas kanal fitur masukan dengan rata-rata penurunan nilai `F1` terbesar pada horizon `t+1` melalui pengujian permutasi terhadap model DeepLabV3+ dengan CA. Berbeda dengan evaluasi dua puluh sampel awal yang rentan mengalami penurunan seragam ke titik nol akibat minimnya piksel api baru, formulasi terbaru mengevaluasi seluruh himpunan data uji secara per *batch* dengan nilai dasar konsisten dan dua kali ulangan acak sehingga kontribusi relatif masing-masing variabel lingkungan serta riwayat api dapat terbedakan dengan jelas setelah sel dijalankan.

# **10. Ablation Study**

Bagian ini membuktikan kontribusi fungsional setiap komponen mekanika fisika dan fungsi kerugian pada arsitektur usulan DeepLabV3+ dengan CA melalui pengujian pelepasan komponen secara sistematis. Hasil pengujian memperlihatkan bahwa penghapusan komponen angin atau lereng, gerbang bahan bakar, maupun istilah kerugian tambahan menyebabkan penurunan performa yang substansial dibandingkan konfigurasi penuh.

* Menetapkan lima skenario perbandingan mulai dari DeepLabV3+ tanpa CA, pelepasan komponen angin dan lereng, pelepasan gerbang bahan bakar, penggunaan *Focal Tversky Loss* saja, hingga konfigurasi penuh yang diusulkan.
* Melatih ulang ketiga varian ablasi selama sepuluh *epoch* dan mengevaluasinya melalui penetapan ambang validasi yang seragam pada data uji.
* Menyajikan ringkasan hasil tabel serta grafik batang perbandingan nilai PR AUC antarhorizon yang menegaskan keunggulan konfigurasi penuh sebesar 47,57 persen, 56,16 persen, dan 56,10 persen.

In [ ]:
# prop_key = 'DeepLabV3+ with CA' if 'DeepLabV3+ with CA' in MODELS else 'DeepLabV3+-CA'
prop_key = 'DeepLabV3+ with CA' if 'DeepLabV3+ with CA' in MODELS else 'DeepLabV3+-CA'
# Mendefinisikan lima konfigurasi studi ablasi guna mengisolasi kontribusi modul CA, pendorong angin serta lereng, gerbang bahan bakar, dan istilah kerugian.
ablation_suite = [
    {'Configuration': 'DeepLabV3+ (Without CA)',           'bench_key': 'DeepLabV3+'},
    {'Configuration': 'DeepLabV3+-CA (No Wind/Slope)',     'ckpt': 'dl_ca_no_ws',   'kwargs': {'use_wind_slope': False}, 'loss': physics_loss},
    {'Configuration': 'DeepLabV3+-CA (No Fuel Gate)',      'ckpt': 'dl_ca_no_fg',   'kwargs': {'use_fuel_gate': False},  'loss': physics_loss},
    {'Configuration': 'DeepLabV3+-CA (Focal-Tversky Only)','ckpt': 'dl_ca_ft_only', 'kwargs': {},                        'loss': partial(physics_loss, w_bce=0.0, w_area=0.0)},
    {'Configuration': 'DeepLabV3+-CA (Proposed Full)',     'bench_key': prop_key}
]

In [ ]:
# Mengevaluasi varian model hasil studi ablasi secara seragam menggunakan kalibrasi ambang validasi terhadap data uji.
def run_eval_pipeline(model):
    vp = predict_dl(model, val_loader)
    tp = predict_dl(model, test_loader)
    taus = find_thresholds(vp, val_d['y_new'], val_d['dom'])
    return pd.DataFrame(calc_metrics(tp, test_d['y_new'], test_d['dom'], taus))

In [ ]:
# Melatih ulang ketiga varian ablasi selama sepuluh epoch dan menyimpan ringkasan perbandingan metrik ke berkas CSV.
ab_rows = []
for cfg in ablation_suite:
    if 'bench_key' in cfg:
        sub = bench_df[bench_df['Model'] == cfg['bench_key']]
    else:
        kwargs = {**cfg['kwargs'], 'is_ca': True}
        m_ab = build_deeplab(**kwargs)
        m_ab, _ = fit_model(cfg['ckpt'], m_ab, train_loader, val_loader, epochs=10, loss_fn=cfg['loss'], retrain=RETRAIN)
        sub = run_eval_pipeline(m_ab)
        del m_ab
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    ab_rows.append({
        'Configuration': cfg['Configuration'],
        'PR-AUC t1 (%)': round(sub[sub['h'] == 1]['PR-AUC'].values[0] * 100, 2),
        'PR-AUC t2 (%)': round(sub[sub['h'] == 2]['PR-AUC'].values[0] * 100, 2),
        'PR-AUC t3 (%)': round(sub[sub['h'] == 3]['PR-AUC'].values[0] * 100, 2),
        'F1 t1': round(sub[sub['h'] == 1]['F1'].values[0], 4),
        'F1 t2': round(sub[sub['h'] == 2]['F1'].values[0], 4),
        'F1 t3': round(sub[sub['h'] == 3]['F1'].values[0], 4),
        'Area Ratio t1': round(sub[sub['h'] == 1]['Area Ratio'].values[0], 2),
        'Area Ratio t2': round(sub[sub['h'] == 2]['Area Ratio'].values[0], 2),
        'Area Ratio t3': round(sub[sub['h'] == 3]['Area Ratio'].values[0], 2)
    })

ab_df = pd.DataFrame(ab_rows)
ab_df.to_csv(RESULT_DIR / 'ablation_results.csv', index=False)
ab_df

In [ ]:
ab_long = ab_df.melt(
    id_vars='Configuration',
    value_vars=['PR-AUC t1 (%)', 'PR-AUC t2 (%)', 'PR-AUC t3 (%)'],
    var_name='Horizon',
    value_name='PR-AUC (%)'
)
ab_long['Horizon'] = ab_long['Horizon'].map({'PR-AUC t1 (%)': 't+1 (24 jam)', 'PR-AUC t2 (%)': 't+2 (48 jam)', 'PR-AUC t3 (%)': 't+3 (72 jam)'})
fig, ax = plt.subplots(figsize=(9.5, 5.0))
sns.barplot(
    data=ab_long, x='Configuration', y='PR-AUC (%)', hue='Horizon',
    palette={'t+1 (24 jam)': SCIENTIFIC_PALETTE[0], 't+2 (48 jam)': SCIENTIFIC_PALETTE[1], 't+3 (72 jam)': SCIENTIFIC_PALETTE[3]},
    edgecolor='white', linewidth=0.7, ax=ax
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', padding=2, fontsize=7.5, fontweight='bold')
ax.set_ylim(0, max(ab_long['PR-AUC (%)'].max() * 1.18, 10.0))
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right', fontsize=8.5)
ax.set_title('Studi Ablasi: Perbandingan PR-AUC Antar Horizon', fontweight='bold')
ax.set_xlabel('Konfigurasi Model')
ax.set_ylabel('PR-AUC (%)')
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.legend(title='Horizon Prediksi', frameon=True, facecolor='white', edgecolor='#e2e8f0')
plt.tight_layout()
plt.show()

Grafik batang berkelompok di atas membandingkan nilai PR AUC pada ketiga horizon prediksi (`t+1`, `t+2`, dan `t+3`) di antara lima varian konfigurasi dalam studi ablasi arsitektur DeepLabV3+ dengan CA. Konfigurasi penuh yang diusulkan meraih capaian PR AUC tertinggi secara konsisten sebesar 47,57 persen, 56,16 persen, dan 56,10 persen yang melampaui varian tanpa modul CA sebesar 41,45 persen, 51,93 persen, dan 54,69 persen. Pelepasan komponen pendorong angin maupun lereng memangkas kinerja ke angka 33,71 persen, 41,61 persen, dan 42,39 persen, penghapusan gerbang bahan bakar menurunkan skor ke kisaran 35,06 persen, 42,75 persen, dan 42,66 persen, serta pemakaian fungsi kerugian *Focal Tversky Only* menjatuhkan PR AUC tajam hingga 23,92 persen, 33,91 persen, dan 36,10 persen, yang menegaskan bahwa sinergi dinamika CA fisik dan keseluruhan istilah kerugian sangat krusial terhadap ketepatan proyeksi multihorizon.

# **11. Statistical Test**

Bagian ini melakukan pengujian hipotesis statistik nonparametrik pada tingkat peristiwa kebakaran guna memastikan konsistensi keunggulan model usulan DeepLabV3+ dengan CA di seluruh kejadian uji. Hasil uji pada horizon pertama membuktikan adanya perbedaan kinerja global yang sangat signifikan antarmodel dengan statistik Friedman sebesar 97,7721 serta nilai peluang sebesar $2{,}9312 \times 10^{-20}$, disertai pengujian lanjutan multihorizon berarah pada tiga horizon waktu untuk menilai signifikansi secara menyeluruh.

* Menghitung nilai PR AUC horizon pertama untuk setiap peristiwa kebakaran secara terpisah di seluruh kelima model.
* Menjalankan uji Friedman global untuk menguji ada tidaknya perbedaan kinerja yang bermakna secara statistik pada keseluruhan himpunan peristiwa.
* Melakukan uji berpasangan Wilcoxon bertanda antara DeepLabV3+ dengan CA terhadap keempat model lainnya disertai koreksi *Holm Bonferroni* pada horizon pertama, yang menunjukkan keunggulan signifikan atas XGBoost pada taraf $\alpha = 0{,}05$.
* Menambahkan uji Wilcoxon berpasangan searah pada seluruh pasangan peristiwa dan horizon (`t+1` hingga `t+3`) untuk mengevaluasi keunggulan konsisten di sepanjang proyeksi 72 jam.

In [ ]:
# Menghitung nilai PR AUC horizon pertama pada tingkat peristiwa kebakaran unik sebagai sampel uji statistik.
def event_scores(preds, targets, doms, events):
    uniq = np.unique(events)
    sc = []
    for ev in uniq:
        idx = np.where(events == ev)[0]
        p_ev = preds[idx, 0][doms[idx, 0]]
        t_ev = targets[idx, 0][doms[idx, 0]]
        if t_ev.sum() > 0:
            sc.append(average_precision_score(t_ev, p_ev))
    return np.array(sc) if sc else np.array([0.01])

In [ ]:
# Menguji ada tidaknya perbedaan kinerja global yang signifikan antar kelima model memakai uji Friedman nonparametrik.
ev_scores_dict = {name: event_scores(test_preds[name], test_d['y_new'], test_d['dom'], test_d['event_id']) for name in test_preds}
try:
    stat_f, p_f = friedmanchisquare(*ev_scores_dict.values())
except Exception:
    stat_f, p_f = 0.0, 1.0
    
print(f'Global Friedman Chi-Square Statistic: {stat_f:.4f} | p-value: {p_f:.4e}')

In [ ]:
# Menjalankan uji berpasangan Wilcoxon bertanda terhadap model usulan disertai koreksi multipel Holm Bonferroni dan mengekspor hasilnya.
ref_model = 'DeepLabV3+ with CA'
ref_scores = ev_scores_dict[ref_model]
pw_rows = []
for name, sc in ev_scores_dict.items():
    if name == ref_model:
        continue
    try:
        stat_w, p_w = (0.0, 1.0) if np.allclose(ref_scores, sc) else wilcoxon(ref_scores, sc, alternative='two-sided')
    except Exception:
        stat_w, p_w = 0.0, 1.0
    pw_rows.append({
        'Comparison': f'{ref_model} vs {name}',
        'Mean_PR_AUC_Diff_%': round(float(np.mean(ref_scores - sc) * 100.0), 3),
        'Wilcoxon_Stat': round(float(stat_w), 4),
        'p_raw': float(p_w)
    })
pw_df = pd.DataFrame(pw_rows).sort_values('p_raw').reset_index(drop=True)
m_tests = len(pw_df)
holm_p = []
running_max = 0.0
for idx, r in pw_df.iterrows():
    adj = min(1.0, r['p_raw'] * (m_tests - idx))
    running_max = max(running_max, adj)
    holm_p.append(round(running_max, 5))
pw_df['p_Holm_Bonferroni'] = holm_p
pw_df['Significant (alpha=0.05)'] = pw_df['p_Holm_Bonferroni'] < 0.05
pw_df.to_csv(RESULT_DIR / "significant_statistical_test.csv", index=False)
pw_df

In [ ]:
# Menghimpun nilai PR AUC untuk setiap pasangan peristiwa dan horizon sepanjang proyeksi 72 jam.
def multihorizon_event_scores(preds, targets, doms, events):
    uniq = np.unique(events)
    sc = []
    for ev in uniq:
        idx = np.where(events == ev)[0]
        for h in range(H):
            p_eh = preds[idx, h][doms[idx, h]]
            t_eh = targets[idx, h][doms[idx, h]]
            if t_eh.sum() > 0:
                sc.append(average_precision_score(t_eh, p_eh))
    return np.array(sc) if sc else np.array([0.01])

# Menguji keunggulan berarah model DeepLabV3+ dengan CA lintas seluruh pasangan peristiwa dan horizon memakai koreksi Holm Bonferroni.
ref_mh_model = 'DeepLabV3+ with CA' if 'DeepLabV3+ with CA' in test_preds else ref_model
mh_scores_dict = {name: multihorizon_event_scores(test_preds[name], test_d['y_new'], test_d['dom'], test_d['event_id']) for name in test_preds}
ref_mh = mh_scores_dict[ref_mh_model]
mh_rows = []
for name, sc in mh_scores_dict.items():
    if name == ref_mh_model:
        continue
    try:
        stat_w, p_w = (0.0, 1.0) if np.allclose(ref_mh, sc) else wilcoxon(ref_mh, sc, alternative='greater')
    except Exception:
        stat_w, p_w = 0.0, 1.0
    mh_rows.append({
        'Comparison': f'{ref_mh_model} vs {name}',
        'N_Pairs': len(ref_mh),
        'Mean_PR_AUC_Diff_%': round(float(np.mean(ref_mh - sc) * 100.0), 3),
        'Wilcoxon_Stat': round(float(stat_w), 4),
        'p_raw': float(p_w),
    })
pw_mh_df = pd.DataFrame(mh_rows).sort_values('p_raw').reset_index(drop=True)
m_mh = len(pw_mh_df)
holm_mh, run_max_mh = [], 0.0
for idx, r in pw_mh_df.iterrows():
    adj = min(1.0, r['p_raw'] * (m_mh - idx))
    run_max_mh = max(run_max_mh, adj)
    holm_mh.append(round(run_max_mh, 5))
pw_mh_df['p_Holm_Bonferroni'] = holm_mh
pw_mh_df['Significant (alpha=0.05)'] = pw_mh_df['p_Holm_Bonferroni'] < 0.05
pw_mh_df.to_csv(RESULT_DIR / 'multihorizon_statistical_test.csv', index=False)
pw_mh_df

# <b><font color='black'>Kesimpulan</font></b>
---

Penelitian ini membuktikan bahwa integrasi jaringan segmentasi spasial dengan persamaan penjalaran fisika *Neural Cellular Automata* mampu memprediksi dinamika perluasan kebakaran hutan selama 72 jam di Pulau Kalimantan secara akurat, terkalibrasi, dan dapat diinterpretasikan secara fisis. Di antara seluruh pendekatan yang diuji pada himpunan data uji independen, arsitektur yang diusulkan, yaitu DeepLabV3+ dengan modul *Cellular Automata*, memberikan hasil terbaik dalam memetakan pertumbuhan tepi api di tengah ketimpangan kelas yang ekstrem dan kompleksitas lingkungan gambut tropis.

* **Pola Risiko dan Faktor Pendorong Kebakaran** dalam temuan eksploratif mengonfirmasi bahwa aktivitas kebakaran terkonsentrasi di wilayah selatan dan barat Kalimantan selama puncak musim kemarau, yaitu Agustus hingga Oktober, serta meningkat pada fase El Niño dengan DMI positif, dengan rasio ketimpangan piksel terbakar baru mencapai sekitar 244 banding 1. Lahan terdeforestasi yang jauh dari akses jalan, arah pergerakan api akibat adveksi angin dominan pada 90 hingga 110 derajat, serta dominasi lahan gambut di Kalimantan Selatan dan Kalimantan Tengah terbukti menjadi faktor utama yang mendorong perluasan area terbakar.

* **Capaian Kinerja Kuantitatif dan Efisiensi Komputasi** menunjukkan bahwa DeepLabV3+ dengan CA memperoleh nilai tertinggi secara konsisten pada 794 sampel uji, dengan PR AUC berturut-turut sebesar 0,4757, 0,5616, dan 0,5610, skor `F1` sebesar 0,5583, 0,6184, dan 0,6145, serta *Brier Score Loss* terendah sebesar 0,0241 pada horizon ketiga. Model ini mengungguli DeepLabV3+, U-Net, U-Net dengan CA, dan XGBoost. Keunggulan kinerja tersebut disertai peningkatan *inference time* menjadi 35,14 milidetik per *patch* akibat iterasi mikro harian, dibandingkan dengan 6,80 milidetik per *patch* pada DeepLabV3+ tanpa CA.

* **Kontribusi Mekanika Fisika dan Fungsi Kerugian** terkonfirmasi melalui studi ablasi yang memperlihatkan penurunan PR AUC secara tajam ketika modul CA, faktor angin dan lereng, *fuel gate*, ataupun komponen *boundary loss* dan *area calibration loss* dihilangkan. Nilainya bahkan turun ke kisaran 23,92 hingga 36,10 persen pada konfigurasi *Focal Tversky Only*. Kombinasi antara kendala dinamika fisika dan *composite loss function* terbukti penting untuk mencegah *overprediction* sekaligus menjaga ketajaman batas penjalaran api.

* **Validasi Statistik dan Evaluasi Kontribusi Fitur** membuktikan adanya perbedaan kinerja global yang sangat signifikan di antara kelima model berdasarkan uji Friedman, dengan statistik 97,7721 dan nilai peluang sebesar $2{,}9312 \times 10^{-20}$. Model yang diusulkan unggul secara signifikan atas XGBoost pada uji *post hoc* horizon pertama dan telah dilengkapi dengan pengujian Wilcoxon berarah pada ketiga horizon. Sementara itu, analisis *feature permutation* telah disempurnakan dari evaluasi terhadap dua puluh sampel awal yang menghasilkan nilai seragam sebesar 0,5583 menjadi evaluasi berkelompok terhadap seluruh sampel uji agar sensitivitas fitur lingkungan dapat diukur secara lebih proporsional.

* **Interpretasi Prediksi Spasial dan Keterbacaan Fisika** memperlihatkan kemampuan model dalam merekonstruksi pola pertumbuhan api konsentris secara realistis serta menghasilkan peta parameter internal *Cellular Automata* yang selaras dengan kondisi medan. Hal ini mencakup aktivasi faktor bahan bakar dan angin yang tinggi pada *fire front*, serta penekanan laju penyebaran di badan sungai. Model juga menunjukkan kestabilan skor `F1` yang jauh lebih baik pada lahan gambut dibandingkan dengan lahan nongambut. Keterbatasan utama muncul pada kejadian kebakaran besar yang melampaui 1.000 hektar dan area yang terhalang tutupan awan tebal. Kondisi tersebut menyebabkan proyeksi luas area terbakar cenderung lebih rendah (*underprediction*), dengan rasio luas sekitar 0,66 hingga 0,69.

* **Rekomendasi Pengembangan dan Penerapan** mencakup integrasi sumber data *remote sensing* yang mampu menembus tutupan awan, seperti citra radar, serta kalibrasi bobot penalti luas pada kejadian ekstrem untuk mengurangi kecenderungan *underprediction*. Selain itu, pengujian ulang *feature permutation* dan uji statistik multihorizon perlu dilakukan di lingkungan GPU Colab untuk memperoleh hasil numerik terbaru. Dalam penerapan sistem peringatan dini di kawasan rawan kebakaran gambut pada puncak musim kemarau, DeepLabV3+ dengan CA direkomendasikan sebagai model utama, sedangkan DeepLabV3+ tanpa CA dapat dipilih ketika *inference latency* di bawah sepuluh milidetik per *patch* menjadi prioritas.

Secara keseluruhan, penelitian ini menunjukkan bahwa integrasi *spatial deep learning* dan prinsip fisika penjalaran api menawarkan kerangka prediksi kebakaran hutan yang lebih tepercaya dan informatif untuk mendukung upaya mitigasi berbasis bukti ilmiah.

---

<div align="center">

  <h4><b>Akhir Laporan Penelitian Prediksi Penyebaran Kebakaran Hutan</b></h4>
</div>